In [ ]:
# CELL 1: Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

In [ ]:
# CELL 2 : Package File: XAI Libraries, Install timm
import timm
!pip install -q grad-cam pytorch-grad-cam
!pip install -q timm
!pip install -q grad-cam
!pip install -q pytorch-grad-cam

print("✅ XAI libraries installed!")
print(f"✅ timm installed: {timm.__version__}")

In [ ]:
# CELL 3: Import All Libraries and Fix Random Seeds

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
from sklearn.preprocessing import label_binarize
from torchvision import transforms
import timm
import time

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
import torchvision.models as models

#GradCAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Image processing
import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
    auc,
    roc_curve
)
from sklearn.model_selection import train_test_split, StratifiedKFold

# Visualization
from matplotlib.gridspec import GridSpec
# ========== FIX ALL RANDOM SEEDS ==========
def seed_everything(seed=42):
    """Fix all random seeds for reproducibility"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✅ All random seeds fixed to {seed}")

# Fix seeds
SEED = 42 #42,123,456,
seed_everything(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n✅ All libraries imported successfully!")
print("✅ Seeds fixed - Results will be reproducible")

In [ ]:
# CELL 4: Dataset Path Setup and Structure Verification
# Define paths

BASE_DIR = Path("/content/drive/MyDrive/knee")
DATA_DIR = BASE_DIR / "data"
ZIP_PATH = BASE_DIR / "data_full.zip"
OUTPUT_DIR = BASE_DIR / "outputs"

print(f"📂 BASE_DIR exists: {BASE_DIR.exists()}")
print(f"📂 DATA_DIR exists: {DATA_DIR.exists()}")
print(f"📦 ZIP exists: {ZIP_PATH.exists()}")
print(f"📂 OUTPUT_DIR exists: {OUTPUT_DIR.exists()}")

# Check outputs/checkpoints
if OUTPUT_DIR.exists():
    print(f"\n✅ Output folder contents:")
    for item in sorted(OUTPUT_DIR.iterdir()):
        if item.is_dir():
            files = list(item.rglob("*"))
            print(f"   📁 {item.name}/ — {len(files)} files")
            # Show .pt or .pth files (saved models)
            model_files = [f for f in files if f.suffix in ['.pt', '.pth']]
            for mf in model_files:
                size_mb = mf.stat().st_size / (1024*1024)
                print(f"      💾 {mf.name} ({size_mb:.1f} MB)")

# Extract zip if data folder missing
if not DATA_DIR.exists() and ZIP_PATH.exists():
    print(f"\n📦 Extracting {ZIP_PATH.name}...")
    import zipfile
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(BASE_DIR)
    print(f"✅ Extraction complete!")
    print(f"📂 DATA_DIR now exists: {DATA_DIR.exists()}")

# Verify data structure after extraction
if DATA_DIR.exists():
    print(f"\n📊 Dataset Structure:")
    splits = sorted([f for f in DATA_DIR.iterdir() if f.is_dir()])
    total_all = 0
    for split in splits:
        grades = sorted([g for g in split.iterdir() if g.is_dir()])
        split_total = 0
        print(f"\n  {split.name.upper()}:")
        for grade in grades:
            imgs = list(grade.glob('*.jpg')) + list(grade.glob('*.png')) + \
                   list(grade.glob('*.jpeg')) + list(grade.glob('*.JPG'))
            print(f"    Grade {grade.name}: {len(imgs)} images")
            split_total += len(imgs)
        print(f"    Subtotal: {split_total}")
        total_all += split_total
    print(f"\n  ✅ TOTAL IMAGES: {total_all}")
else:
    print(f"\n❌ Data still not found after extraction attempt.")
    print(f"   ZIP contents preview:")
    if ZIP_PATH.exists():
        import zipfile
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            names = z.namelist()[:20]
            for n in names:
                print(f"      {n}")

In [ ]:
# CELL 5 (UPDATED): Deep Dataset Exploration - Nested Folders

def explore_nested_dataset(data_dir):
    """
    Explore nested dataset structure (train/test/val -> Grade folders -> images)
    """
    print("🔍 Exploring NESTED dataset structure...\n")

    all_data = []

    # Level 1: train/test/val folders
    for split_folder in sorted(data_dir.iterdir()):
        if split_folder.is_dir():
            print(f"{'='*60}")
            print(f"📁 SPLIT: {split_folder.name}")
            print(f"{'='*60}")

            split_total = 0

            # Level 2: Grade folders inside each split
            for grade_folder in sorted(split_folder.iterdir()):
                if grade_folder.is_dir():
                    # Get all image files
                    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.JPG', '*.PNG']
                    images = []
                    for ext in image_extensions:
                        images.extend(list(grade_folder.glob(ext)))

                    num_images = len(images)
                    split_total += num_images

                    print(f"   📂 {grade_folder.name}: {num_images} images")

                    # Sample image info (if images exist)
                    sample_size = None
                    sample_mode = None
                    if images:
                        try:
                            sample_img = Image.open(images[0])
                            sample_size = sample_img.size
                            sample_mode = sample_img.mode
                        except:
                            pass

                    # Store info
                    all_data.append({
                        'split': split_folder.name,
                        'grade': grade_folder.name,
                        'num_images': num_images,
                        'image_size': sample_size,
                        'image_mode': sample_mode
                    })

            print(f"   {'─'*50}")
            print(f"   TOTAL in {split_folder.name}: {split_total} images\n")

    # Create summary dataframe
    df_summary = pd.DataFrame(all_data)

    return df_summary

# Explore nested dataset
df_dataset = explore_nested_dataset(DATA_DIR)

# Display full summary
print("\n" + "="*60)
print("📊 COMPLETE DATASET SUMMARY")
print("="*60)
print(df_dataset.to_string(index=False))

# Summary statistics
print("\n" + "="*60)
print("📈 STATISTICS")
print("="*60)
print("\nImages per split:")
print(df_dataset.groupby('split')['num_images'].sum())

print("\nImages per grade (across all splits):")
print(df_dataset.groupby('grade')['num_images'].sum().sort_index())

print(f"\n🎯 TOTAL DATASET SIZE: {df_dataset['num_images'].sum()} images")

# Check for class imbalance
grade_counts = df_dataset.groupby('grade')['num_images'].sum()
if len(grade_counts) > 0:
    print(f"\n⚖️ Class Balance:")
    print(f"   Most common: {grade_counts.idxmax()} ({grade_counts.max()} images)")
    print(f"   Least common: {grade_counts.idxmin()} ({grade_counts.min()} images)")
    print(f"   Imbalance ratio: {grade_counts.max() / grade_counts.min():.2f}x")

print("\n✅ Nested dataset exploration complete!")

In [ ]:
# CELL 6 (UPDATED): Visualize Sample Images from Each Grade

def visualize_nested_samples(data_dir, split='train', samples_per_grade=3):
    """
    Visualize sample images from each grade folder within a split
    """
    split_path = data_dir / split

    if not split_path.exists():
        print(f"❌ Split '{split}' not found!")
        return

    # Get all grade folders
    grade_folders = sorted([f for f in split_path.iterdir() if f.is_dir()])

    if len(grade_folders) == 0:
        print(f"❌ No grade folders found in '{split}'!")
        return

    print(f"🖼️ Visualizing samples from '{split}' split...\n")

    # Create subplot grid
    fig, axes = plt.subplots(len(grade_folders), samples_per_grade,
                             figsize=(15, 4*len(grade_folders)))

    if len(grade_folders) == 1:
        axes = axes.reshape(1, -1)

    for row_idx, grade_folder in enumerate(grade_folders):
        # Get image files
        image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.JPG', '*.PNG']
        images = []
        for ext in image_extensions:
            images.extend(list(grade_folder.glob(ext)))

        if len(images) == 0:
            print(f"⚠️ No images in {grade_folder.name}")
            continue

        # Sample random images
        sample_images = random.sample(images, min(samples_per_grade, len(images)))

        for col_idx, img_path in enumerate(sample_images):
            try:
                img = Image.open(img_path)

                # Convert to RGB if grayscale for consistent display
                if img.mode == 'L':
                    axes[row_idx, col_idx].imshow(img, cmap='gray')
                else:
                    axes[row_idx, col_idx].imshow(img)

                axes[row_idx, col_idx].axis('off')

                # Title
                if col_idx == 0:
                    axes[row_idx, col_idx].set_title(
                        f"{grade_folder.name}\n{img.size} | {img.mode}",
                        fontsize=10,
                        loc='left',
                        fontweight='bold'
                    )
                else:
                    axes[row_idx, col_idx].set_title(
                        f"{img.size}",
                        fontsize=8
                    )
            except Exception as e:
                print(f"⚠️ Error loading {img_path.name}: {e}")

    plt.suptitle(f"Sample Images from '{split.upper()}' Split",
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(f'dataset_samples_{split}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"✅ Visualization saved as 'dataset_samples_{split}.png'")

# Visualize samples from train split
visualize_nested_samples(DATA_DIR, split='train', samples_per_grade=3)

In [ ]:
# CELL 7: Visualize Class Distribution Across Splits

def plot_class_distribution(df_dataset):
    """
    Plot class distribution across train/test/val splits
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Plot 1: Stacked bar chart by split
    pivot_data = df_dataset.pivot_table(
        index='grade',
        columns='split',
        values='num_images',
        fill_value=0
    )

    pivot_data.plot(kind='bar', stacked=False, ax=axes[0],
                    color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
    axes[0].set_title('Images per Grade Across Splits', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Grade', fontsize=12)
    axes[0].set_ylabel('Number of Images', fontsize=12)
    axes[0].legend(title='Split', fontsize=10)
    axes[0].grid(axis='y', alpha=0.3)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

    # Plot 2: Pie chart - total per grade
    grade_totals = df_dataset.groupby('grade')['num_images'].sum()
    colors = plt.cm.Set3(range(len(grade_totals)))

    axes[1].pie(grade_totals, labels=grade_totals.index, autopct='%1.1f%%',
                startangle=90, colors=colors, textprops={'fontsize': 11})
    axes[1].set_title('Overall Grade Distribution', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("✅ Class distribution saved as 'class_distribution.png'")

# Plot distribution
plot_class_distribution(df_dataset)

# Print detailed distribution
print("\n📊 Detailed Distribution Table:")
pivot_table = df_dataset.pivot_table(
    index='grade',
    columns='split',
    values='num_images',
    fill_value=0,
    margins=True,
    margins_name='TOTAL'
)
print(pivot_table)

In [ ]:
# CELL 8: Custom Dataset Class for Knee OA

class KneeOADataset(Dataset):
    """
    Custom PyTorch Dataset for Knee Osteoarthritis Classification
    """
    def __init__(self, data_dir, split='train', transform=None):
        """
        Args:
            data_dir: Path to data directory (E:/knee/data)
            split: 'train', 'val', 'test', or 'auto_test'
            transform: Albumentations transforms
        """
        self.data_dir = Path(data_dir) / split
        self.transform = transform
        self.split = split

        # Collect all image paths and labels
        self.images = []
        self.labels = []
        self.grade_to_idx = {}

        # Get all grade folders
        grade_folders = sorted([f for f in self.data_dir.iterdir() if f.is_dir()])

        for idx, grade_folder in enumerate(grade_folders):
            grade_name = grade_folder.name
            self.grade_to_idx[grade_name] = idx

            # Get all images in this grade
            image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.JPG', '*.PNG']
            for ext in image_extensions:
                for img_path in grade_folder.glob(ext):
                    self.images.append(str(img_path))
                    self.labels.append(idx)

        self.idx_to_grade = {v: k for k, v in self.grade_to_idx.items()}

        print(f"✅ Loaded {split} split:")
        print(f"   Total images: {len(self.images)}")
        print(f"   Classes: {self.grade_to_idx}")
        print(f"   Class distribution: {np.bincount(self.labels)}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load image
        img_path = self.images[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Convert to RGB (3 channels) for pretrained models
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

        label = self.labels[idx]

        # Apply transforms
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, label

    def get_class_counts(self):
        """Return counts for each class"""
        return np.bincount(self.labels)

    def get_class_weights(self):
        """Calculate class weights for balanced loss"""
        counts = self.get_class_counts()
        weights = 1.0 / counts
        weights = weights / weights.sum() * len(counts)  # Normalize
        return torch.FloatTensor(weights)


# Test dataset loading
print("📦 Testing dataset loading...\n")

# Load train dataset (without transforms for now)
train_dataset = KneeOADataset(DATA_DIR, split='train', transform=None)

# Load validation dataset
val_dataset = KneeOADataset(DATA_DIR, split='val', transform=None)

# Load test dataset (merge test + auto_test)
test_dataset = KneeOADataset(DATA_DIR, split='test', transform=None)

print("\n" + "="*60)
print("📊 Dataset Summary:")
print("="*60)
print(f"Train: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")
print(f"Test:  {len(test_dataset)} images")
print(f"Total: {len(train_dataset) + len(val_dataset) + len(test_dataset)} images")

# Calculate class weights for weighted loss
class_weights = train_dataset.get_class_weights()
print("\n⚖️ Class Weights (for handling imbalance):")
for idx, weight in enumerate(class_weights):
    grade = train_dataset.idx_to_grade[idx]
    print(f"   Grade {grade}: {weight:.4f}")

print("\n✅ Dataset class created successfully!")

In [ ]:
# CELL 9: Data Augmentation and Preprocessing Transforms

def get_transforms(split='train', img_size=224):
    """
    Get augmentation transforms for train/val/test

    Args:
        split: 'train', 'val', or 'test'
        img_size: Target image size (default 224)

    Returns:
        Albumentations compose transform
    """

    if split == 'train':
        # Training augmentation - aggressive but medically valid
        transform = A.Compose([
            A.Resize(img_size, img_size),

            # Geometric transforms
            A.HorizontalFlip(p=0.5),  # Knee X-rays are symmetric
            A.Rotate(limit=15, p=0.5),  # Small rotations
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.1,
                rotate_limit=10,
                p=0.5
            ),

            # Contrast/Brightness (important for X-rays)
            A.CLAHE(clip_limit=2.0, p=0.7),  # Enhance contrast
            A.RandomBrightnessContrast(
                brightness_limit=0.2,
                contrast_limit=0.2,
                p=0.5
            ),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),

            # Noise (simulate imaging artifacts)
            A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),

            # Normalization
            A.Normalize(
                mean=[0.485, 0.456, 0.406],  # ImageNet stats
                std=[0.229, 0.224, 0.225],
                max_pixel_value=255.0
            ),
            ToTensorV2()
        ])

    else:  # val or test - no augmentation
        transform = A.Compose([
            A.Resize(img_size, img_size),
            A.CLAHE(clip_limit=2.0, p=1.0),  # Only contrast enhancement
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
                max_pixel_value=255.0
            ),
            ToTensorV2()
        ])

    return transform


# Create transforms
train_transform = get_transforms('train')
val_transform = get_transforms('val')
test_transform = get_transforms('test')

print("✅ Transforms created:")
print(f"   Train: {len(train_transform)} augmentations")
print(f"   Val:   {len(val_transform)} augmentations")
print(f"   Test:  {len(test_transform)} augmentations")


# Visualize augmentation examples
def visualize_augmentations(dataset, transform, num_samples=3):
    """Show original vs augmented images"""
    fig, axes = plt.subplots(num_samples, 5, figsize=(18, 3*num_samples))

    # Pick random samples
    indices = random.sample(range(len(dataset)), num_samples)

    for row, idx in enumerate(indices):
        # Get original image
        img_path = dataset.images[idx]
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        original_rgb = cv2.cvtColor(original, cv2.COLOR_GRAY2RGB)

        label = dataset.labels[idx]
        grade = dataset.idx_to_grade[label]

        # Show original
        axes[row, 0].imshow(original, cmap='gray')
        axes[row, 0].set_title(f'Original\nGrade {grade}', fontsize=10)
        axes[row, 0].axis('off')

        # Show 4 augmented versions
        for col in range(1, 5):
            augmented = transform(image=original_rgb)['image']

            # Convert tensor to numpy for display
            aug_img = augmented.permute(1, 2, 0).numpy()
            aug_img = (aug_img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
            aug_img = np.clip(aug_img, 0, 1)

            axes[row, col].imshow(aug_img)
            axes[row, col].set_title(f'Aug #{col}', fontsize=10)
            axes[row, col].axis('off')

    plt.tight_layout()
    save_path = OUTPUT_DIR / 'augmentation_examples.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✅ Saved to: {save_path}")
    plt.show()
    print("✅ Augmentation visualization saved as 'augmentation_examples.png'")


# Visualize augmentations on train data
print("\n🖼️ Generating augmentation examples...")
visualize_augmentations(train_dataset, train_transform, num_samples=3)

In [ ]:
# CELL 10 (MODIFIED): Create DataLoaders

# Recreate datasets with transforms
train_dataset_aug = KneeOADataset(DATA_DIR, split='train', transform=train_transform)
val_dataset_aug = KneeOADataset(DATA_DIR, split='val', transform=val_transform)
test_dataset_aug = KneeOADataset(DATA_DIR, split='test', transform=test_transform)

# Hyperparameters
BATCH_SIZE = 32
NUM_WORKERS = 0  # ← CHANGED from 4 to 0 (Windows fix)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset_aug,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,  # 0 = single process (Windows compatible)
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=False  # ← ADDED
)

val_loader = DataLoader(
    val_dataset_aug,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=False
)

test_loader = DataLoader(
    test_dataset_aug,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=False
)

print("✅ DataLoaders created:")
print(f"   Train batches: {len(train_loader)} (batch_size={BATCH_SIZE})")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

# Test dataloader
print("\n🧪 Testing DataLoader...")
for images, labels in train_loader:
    print(f"   Batch shape: {images.shape}")
    print(f"   Labels shape: {labels.shape}")
    print(f"   Image range: [{images.min():.3f}, {images.max():.3f}]")
    print(f"   Label samples: {labels[:10].tolist()}")
    break

print("\n✅ DataLoaders working perfectly!")

# Save class weights for loss function
class_weights = train_dataset_aug.get_class_weights().to(device)
print(f"\n⚖️ Class weights moved to {device}")

In [ ]:
# CELL 11: EfficientNet-B0 Model Definition
class EfficientNetB0(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=pretrained,
            num_classes=0
        )
        feature_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

efficientnet_model = EfficientNetB0(num_classes=5, pretrained=True).to(device)

total_params = sum(p.numel() for p in efficientnet_model.parameters())
trainable_params = sum(p.numel() for p in efficientnet_model.parameters() if p.requires_grad)
print(f"✅ EfficientNet-B0 created")
print(f"   Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.1f}M)")

In [ ]:
# CELL 12: Training Setup for EfficientNet
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    efficientnet_model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30,
    eta_min=1e-6
)

PATIENCE = 15

print(f"✅ Training setup complete")
print(f"   Loss: Weighted Cross-Entropy")
print(f"   Class weights: {class_weights.cpu().numpy()}")
print(f"   Optimizer: AdamW (lr=1e-4)")
print(f"   Scheduler: Cosine Annealing")
print(f"   Patience: {PATIENCE}")

In [ ]:
# CELL 13: Training and Validation Functions
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation', leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return epoch_loss, epoch_acc, qwk, precision, recall, f1, all_preds, all_labels

print("✅ Training functions defined")

In [ ]:
# CELL 14: Main Training Loop for EfficientNet

NUM_EPOCHS = 30
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_qwk': [],
    'lr': []
}

best_val_qwk = 0.0
best_epoch = 0
patience_counter = 0

checkpoint_dir = OUTPUT_DIR / 'checkpoints_baselines'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("="*60)
print("🚀 STARTING EFFICIENTNET-B0 TRAINING")
print("="*60)
print(f"Model: EfficientNet-B0")
print(f"Device: {device}")
print(f"Parameters: {trainable_params/1e6:.1f}M")
print(f"Training samples: {len(train_dataset_aug)}")
print(f"Validation samples: {len(val_dataset_aug)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {NUM_EPOCHS}")
print(f"Seed: {SEED}")
print("="*60)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning Rate: {current_lr:.6f}")

    train_loss, train_acc = train_one_epoch(
        efficientnet_model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc, val_qwk, precision, recall, f1, _, _ = validate(
        efficientnet_model, val_loader, criterion, device
    )

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)
    history['lr'].append(current_lr)

    epoch_time = time.time() - epoch_start
    print(f"\n📊 Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"   Val QWK:    {val_qwk:.4f}")
    print(f"\n📈 Per-Class F1-Scores:")
    for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
        print(f"   Grade {i}: P={p:.3f} R={r:.3f} F1={f:.3f}")
    print(f"\n⏱️ Epoch time: {epoch_time:.1f}s")

    if val_qwk > best_val_qwk:
        best_val_qwk = val_qwk
        best_epoch = epoch + 1
        patience_counter = 0

        checkpoint_path = checkpoint_dir / 'best_efficientnet_model.pth'

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': efficientnet_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_qwk': best_val_qwk,
            'history': history,
            'seed': SEED
        }, checkpoint_path)

        print(f"\n💾 Best model saved! (QWK: {best_val_qwk:.4f})")
    else:
        patience_counter += 1
        print(f"\n⏸️ No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n🛑 Early stopping at epoch {epoch+1}")
        print(f"   Best QWK: {best_val_qwk:.4f} at epoch {best_epoch}")
        break

total_time = time.time() - start_time
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)

print("\n" + "="*60)
print("✅ EFFICIENTNET TRAINING COMPLETE!")
print("="*60)
print(f"Total time: {hours}h {minutes}m")
print(f"Best validation QWK: {best_val_qwk:.4f} (epoch {best_epoch})")
print(f"Best model saved at: {checkpoint_path}")
print(f"Filename: best_efficientnet_model.pth")
print("="*60)

In [ ]:
# CELL 15: ConvNeXt Model Definition
class ConvNeXtModel(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'convnext_base',
            pretrained=pretrained,
            num_classes=0
        )
        feature_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

convnext_model = ConvNeXtModel(num_classes=5, pretrained=True).to(device)

total_params = sum(p.numel() for p in convnext_model.parameters())
trainable_params = sum(p.numel() for p in convnext_model.parameters() if p.requires_grad)
print(f"✅ ConvNeXt-Base created")
print(f"   Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.1f}M)")

In [ ]:
# CELL 16: Training Setup for ConvNeXt
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    convnext_model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30,
    eta_min=1e-6
)

PATIENCE = 15

print(f"✅ Training setup complete")
print(f"   Loss: Weighted Cross-Entropy")
print(f"   Class weights: {class_weights.cpu().numpy()}")
print(f"   Optimizer: AdamW (lr=1e-4)")
print(f"   Scheduler: Cosine Annealing")
print(f"   Patience: {PATIENCE}")

In [ ]:
# CELL 17: Training Loop for ConvNeXt

NUM_EPOCHS = 30
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_qwk': [],
    'lr': []
}

best_val_qwk = 0.0
best_epoch = 0
patience_counter = 0

checkpoint_dir = OUTPUT_DIR / 'checkpoints_baselines'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("="*60)
print("🚀 STARTING CONVNEXT TRAINING")
print("="*60)
print(f"Model: ConvNeXt-Base")
print(f"Device: {device}")
print(f"Parameters: {trainable_params/1e6:.1f}M")
print(f"Training samples: {len(train_dataset_aug)}")
print(f"Validation samples: {len(val_dataset_aug)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {NUM_EPOCHS}")
print(f"Seed: {SEED}")
print("="*60)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning Rate: {current_lr:.6f}")

    train_loss, train_acc = train_one_epoch(
        convnext_model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc, val_qwk, precision, recall, f1, _, _ = validate(
        convnext_model, val_loader, criterion, device
    )

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)
    history['lr'].append(current_lr)

    epoch_time = time.time() - epoch_start
    print(f"\n📊 Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"   Val QWK:    {val_qwk:.4f}")
    print(f"\n📈 Per-Class F1-Scores:")
    for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
        print(f"   Grade {i}: P={p:.3f} R={r:.3f} F1={f:.3f}")
    print(f"\n⏱️ Epoch time: {epoch_time:.1f}s")

    if val_qwk > best_val_qwk:
        best_val_qwk = val_qwk
        best_epoch = epoch + 1
        patience_counter = 0

        checkpoint_path = checkpoint_dir / 'best_convnext_model.pth'

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': convnext_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_qwk': best_val_qwk,
            'history': history,
            'seed': SEED
        }, checkpoint_path)

        print(f"\n💾 Best model saved! (QWK: {best_val_qwk:.4f})")
    else:
        patience_counter += 1
        print(f"\n⏸️ No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n🛑 Early stopping at epoch {epoch+1}")
        print(f"   Best QWK: {best_val_qwk:.4f} at epoch {best_epoch}")
        break

total_time = time.time() - start_time
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)

print("\n" + "="*60)
print("✅ CONVNEXT TRAINING COMPLETE!")
print("="*60)
print(f"Total time: {hours}h {minutes}m")
print(f"Best validation QWK: {best_val_qwk:.4f} (epoch {best_epoch})")
print(f"Best model saved at: {checkpoint_path}")
print(f"Filename: best_convnext_model.pth")
print("="*60)

In [ ]:
# CELL 18: ResNet-50 Baseline Model

class ResNet50Baseline(nn.Module):
    """
    ResNet-50 baseline for Knee OA classification
    """
    def __init__(self, num_classes=5, pretrained=True):
        super(ResNet50Baseline, self).__init__()

        # Load pretrained ResNet-50
        self.backbone = models.resnet50(pretrained=pretrained)

        # Get number of features from last layer
        num_features = self.backbone.fc.in_features

        # Replace final layer
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)


# Initialize model
print("🔧 Initializing ResNet-50 model...")
model = ResNet50Baseline(num_classes=5, pretrained=True)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model initialized:")
print(f"   Architecture: ResNet-50")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Device: {device}")

# Model summary
print(f"\n📊 Model Structure:")
print(f"   Input: [batch_size, 3, 224, 224]")
print(f"   Backbone: ResNet-50 (pretrained on ImageNet)")
print(f"   Classifier: FC(2048→512→5)")
print(f"   Output: [batch_size, 5] (logits)")

# Test forward pass
print(f"\n🧪 Testing forward pass...")
dummy_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    output = model(dummy_input)
print(f"   Input shape: {dummy_input.shape}")
print(f"   Output shape: {output.shape}")
print(f"   Output sample: {output[0].cpu().numpy()}")

print(f"\n✅ Model ready for training!")

In [ ]:
# CELL 19: Loss Function, Optimizer, and Scheduler
# Loss function with class weights (handles imbalance)
criterion = nn.CrossEntropyLoss(weight=class_weights)
criterion = criterion.to(device)

print("📊 Loss Function:")
print(f"   Type: Weighted Cross-Entropy")
print(f"   Class weights: {class_weights.cpu().numpy()}")

# Optimizer - AdamW with weight decay
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print(f"\n⚙️ Optimizer:")
print(f"   Type: AdamW")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Weight decay: {WEIGHT_DECAY}")

# Learning rate scheduler - Cosine Annealing
NUM_EPOCHS = 50
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

print(f"\n📉 LR Scheduler:")
print(f"   Type: Cosine Annealing")
print(f"   Max epochs: {NUM_EPOCHS}")
print(f"   Min LR: 1e-6")

# Early stopping parameters
PATIENCE = 15
best_val_loss = float('inf')
patience_counter = 0

print(f"\n⏸️ Early Stopping:")
print(f"   Patience: {PATIENCE} epochs")

print(f"\n✅ Training setup complete!")

In [ ]:
# CELL 20: Training and Validation Functions

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # Update progress bar
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    # Calculate metrics
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Validate model"""
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation', leave=False):
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Statistics
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    # Calculate per-class metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )

    # Calculate QWK
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

    return epoch_loss, epoch_acc, qwk, precision, recall, f1, all_preds, all_labels


print("✅ Training and validation functions defined!")
print("\n📋 Functions available:")
print("   • train_one_epoch() - Train for one epoch")
print("   • validate() - Validate and compute metrics")

In [ ]:
# CELL 21: Main Training Loop

NUM_EPOCHS = 30
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_qwk': [],
    'lr': []
}

best_val_qwk = 0.0
best_epoch = 0
patience_counter = 0

checkpoint_dir = OUTPUT_DIR / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("="*60)
print("🚀 STARTING TRAINING")
print("="*60)
print(f"Model: ResNet-50 Baseline")
print(f"Device: {device}")
print(f"Training samples: {len(train_dataset_aug)}")
print(f"Validation samples: {len(val_dataset_aug)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {NUM_EPOCHS}")
print(f"Early stopping patience: {PATIENCE}")
print(f"Seed: 456")  # ← এখানে 123
print("="*60)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning Rate: {current_lr:.6f}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc, val_qwk, precision, recall, f1, _, _ = validate(
        model, val_loader, criterion, device
    )

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)
    history['lr'].append(current_lr)

    epoch_time = time.time() - epoch_start
    print(f"\n📊 Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"   Val QWK:    {val_qwk:.4f}")
    print(f"\n📈 Per-Class F1-Scores:")
    for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
        print(f"   Grade {i}: P={p:.3f} R={r:.3f} F1={f:.3f}")
    print(f"\n⏱️ Epoch time: {epoch_time:.1f}s")

    if val_qwk > best_val_qwk:
        best_val_qwk = val_qwk
        best_epoch = epoch + 1
        patience_counter = 0

        checkpoint_path = checkpoint_dir / 'best_model_seed123.pth'  # ← এখানে 123

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_qwk': best_val_qwk,
            'history': history,
            'seed': 456  # ← এখানে 123
        }, checkpoint_path)

        print(f"\n💾 Best model saved! (QWK: {best_val_qwk:.4f})")
    else:
        patience_counter += 1
        print(f"\n⏸️ No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n🛑 Early stopping triggered at epoch {epoch+1}")
        print(f"   Best QWK: {best_val_qwk:.4f} at epoch {best_epoch}")
        break

total_time = time.time() - start_time
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
print(f"Total time: {hours}h {minutes}m")
print(f"Best validation QWK: {best_val_qwk:.4f} (epoch {best_epoch})")
print(f"Best model saved at: {checkpoint_path}")
print(f"Filename: best_model_seed456.pth")  # 123
print("="*60)

In [ ]:
# CELL 22: DenseNet-161 Model Definition

class DenseNet161(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()
        self.backbone = models.densenet161(pretrained=pretrained)
        feature_dim = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

densenet_model = DenseNet161(num_classes=5, pretrained=True).to(device)

total_params = sum(p.numel() for p in densenet_model.parameters())
trainable_params = sum(p.numel() for p in densenet_model.parameters() if p.requires_grad)
print(f"✅ DenseNet-161 created")
print(f"   Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.1f}M)")

In [ ]:
# CELL 23: Training Setup for DenseNet
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    densenet_model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30,
    eta_min=1e-6
)

PATIENCE = 15

print(f"✅ Training setup complete")
print(f"   Loss: Weighted Cross-Entropy")
print(f"   Class weights: {class_weights.cpu().numpy()}")
print(f"   Optimizer: AdamW (lr=1e-4)")
print(f"   Scheduler: Cosine Annealing")
print(f"   Patience: {PATIENCE}")

In [ ]:
# CELL 24: Training and Validation Functions
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation', leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return epoch_loss, epoch_acc, qwk, precision, recall, f1, all_preds, all_labels

print("✅ Training functions defined")

In [ ]:
# CELL 25: Main Training Loop for DenseNet
NUM_EPOCHS = 30

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_qwk': [],
    'lr': []
}

best_val_qwk = 0.0
best_epoch = 0
patience_counter = 0

checkpoint_dir = OUTPUT_DIR / 'checkpoints_baselines'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("="*60)
print("🚀 STARTING DENSENET-161 TRAINING")
print("="*60)
print(f"Model: DenseNet-161")
print(f"Device: {device}")
print(f"Parameters: {trainable_params/1e6:.1f}M")
print(f"Training samples: {len(train_dataset_aug)}")
print(f"Validation samples: {len(val_dataset_aug)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {NUM_EPOCHS}")
print(f"Seed: {SEED}")
print("="*60)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning Rate: {current_lr:.6f}")

    train_loss, train_acc = train_one_epoch(
        densenet_model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc, val_qwk, precision, recall, f1, _, _ = validate(
        densenet_model, val_loader, criterion, device
    )

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)
    history['lr'].append(current_lr)

    epoch_time = time.time() - epoch_start
    print(f"\n📊 Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"   Val QWK:    {val_qwk:.4f}")
    print(f"\n📈 Per-Class F1-Scores:")
    for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
        print(f"   Grade {i}: P={p:.3f} R={r:.3f} F1={f:.3f}")
    print(f"\n⏱️ Epoch time: {epoch_time:.1f}s")

    if val_qwk > best_val_qwk:
        best_val_qwk = val_qwk
        best_epoch = epoch + 1
        patience_counter = 0

        checkpoint_path = checkpoint_dir / 'best_densenet_model.pth'

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': densenet_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_qwk': best_val_qwk,
            'history': history,
            'seed': SEED
        }, checkpoint_path)

        print(f"\n💾 Best model saved! (QWK: {best_val_qwk:.4f})")
    else:
        patience_counter += 1
        print(f"\n⏸️ No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n🛑 Early stopping at epoch {epoch+1}")
        print(f"   Best QWK: {best_val_qwk:.4f} at epoch {best_epoch}")
        break

total_time = time.time() - start_time
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)

print("\n" + "="*60)
print("✅ DENSENET TRAINING COMPLETE!")
print("="*60)
print(f"Total time: {hours}h {minutes}m")
print(f"Best validation QWK: {best_val_qwk:.4f} (epoch {best_epoch})")
print(f"Best model saved at: {checkpoint_path}")
print(f"Filename: best_densenet_model.pth")
print("="*60)

In [ ]:
# CELL 26: Swin Transformer Model Definition

class SwinTransformer(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'swin_base_patch4_window7_224',
            pretrained=pretrained,
            num_classes=0  # Remove head
        )
        feature_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

swin_model = SwinTransformer(num_classes=5, pretrained=True).to(device)

total_params = sum(p.numel() for p in swin_model.parameters())
trainable_params = sum(p.numel() for p in swin_model.parameters() if p.requires_grad)
print(f"✅ Swin Transformer created")
print(f"   Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.1f}M)")

In [ ]:
# CELL 27: Training Setup for Swin
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    swin_model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30,
    eta_min=1e-6
)

PATIENCE = 15

print(f"✅ Training setup complete")
print(f"   Loss: Weighted Cross-Entropy")
print(f"   Class weights: {class_weights.cpu().numpy()}")
print(f"   Optimizer: AdamW (lr=1e-4)")
print(f"   Scheduler: Cosine Annealing")
print(f"   Patience: {PATIENCE}")

In [ ]:
# CELL 28: Training and Validation Functions

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        # Forward
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()
        optimizer.step()

        # Stats
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Validate model"""
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation', leave=False):
            images = images.to(device)
            labels = labels.to(device)

            # Forward
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Stats
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Metrics
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )

    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

    return epoch_loss, epoch_acc, qwk, precision, recall, f1, all_preds, all_labels

print("✅ Training functions defined")

In [ ]:
# CELL 29: Main Training Loop for Swin
NUM_EPOCHS = 30

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_qwk': [],
    'lr': []
}

best_val_qwk = 0.0
best_epoch = 0
patience_counter = 0

checkpoint_dir = OUTPUT_DIR / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("="*60)
print("🚀 STARTING SWIN TRANSFORMER TRAINING")
print("="*60)
print(f"Model: Swin-Base")
print(f"Device: {device}")
print(f"Total parameters: {total_params/1e6:.1f}M")
print(f"Trainable parameters: {trainable_params/1e6:.1f}M")
print(f"Training samples: {len(train_dataset_aug)}")
print(f"Validation samples: {len(val_dataset_aug)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {NUM_EPOCHS}")
print(f"Seed: {SEED}")
print("="*60)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Learning Rate: {current_lr:.6f}")

    train_loss, train_acc = train_one_epoch(
        swin_model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc, val_qwk, precision, recall, f1, _, _ = validate(
        swin_model, val_loader, criterion, device
    )

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)
    history['lr'].append(current_lr)

    epoch_time = time.time() - epoch_start
    print(f"\n📊 Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"   Val QWK:    {val_qwk:.4f}")
    print(f"\n📈 Per-Class F1-Scores:")
    for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
        print(f"   Grade {i}: P={p:.3f} R={r:.3f} F1={f:.3f}")
    print(f"\n⏱️ Epoch time: {epoch_time:.1f}s")

    if val_qwk > best_val_qwk:
        best_val_qwk = val_qwk
        best_epoch = epoch + 1
        patience_counter = 0

        checkpoint_path = checkpoint_dir / 'best_swin_model.pth'

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': swin_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_qwk': best_val_qwk,
            'history': history,
            'seed': SEED
        }, checkpoint_path)

        print(f"\n💾 Best model saved! (QWK: {best_val_qwk:.4f})")
    else:
        patience_counter += 1
        print(f"\n⏸️ No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n🛑 Early stopping at epoch {epoch+1}")
        print(f"   Best QWK: {best_val_qwk:.4f} at epoch {best_epoch}")
        break

total_time = time.time() - start_time
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)

print("\n" + "="*60)
print("✅ SWIN TRAINING COMPLETE!")
print("="*60)
print(f"Total time: {hours}h {minutes}m")
print(f"Best validation QWK: {best_val_qwk:.4f} (epoch {best_epoch})")
print(f"Best model saved: best_swin_model.pth")
print("="*60)

In [ ]:
# CELL 30: Visualize Training History

def plot_training_history(history, save_dir):
    """Plot training curves"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    epochs = range(1, len(history['train_loss']) + 1)

    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2, marker='o', markersize=4)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2, marker='s', markersize=4)
    axes[0, 0].set_xlabel('Epoch', fontsize=14, fontweight='bold')
    axes[0, 0].set_ylabel('Loss', fontsize=14, fontweight='bold')
    axes[0, 0].set_title('Training and Validation Loss', fontsize=16, fontweight='bold')
    axes[0, 0].legend(fontsize=13)
    axes[0, 0].grid(alpha=0.3)

    # Accuracy
    axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2, marker='o', markersize=4)
    axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2, marker='s', markersize=4)
    axes[0, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    axes[0, 1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    axes[0, 1].legend(fontsize=11)
    axes[0, 1].grid(alpha=0.3)

    # QWK
    axes[1, 0].plot(epochs, history['val_qwk'], 'g-', linewidth=2, marker='o', markersize=6)
    axes[1, 0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Quadratic Weighted Kappa', fontsize=12, fontweight='bold')
    axes[1, 0].set_title('Validation QWK Score', fontsize=14, fontweight='bold')
    axes[1, 0].grid(alpha=0.3)

    # Mark best epoch
    best_qwk = max(history['val_qwk'])
    best_epoch = history['val_qwk'].index(best_qwk) + 1
    axes[1, 0].axvline(x=best_epoch, color='r', linestyle='--', linewidth=2, alpha=0.7)
    axes[1, 0].axhline(y=best_qwk, color='r', linestyle='--', linewidth=2, alpha=0.7,
                       label=f'Best: {best_qwk:.4f} (Epoch {best_epoch})')
    axes[1, 0].legend(fontsize=11)

    # Learning Rate
    axes[1, 1].plot(epochs, history['lr'], 'purple', linewidth=2, marker='d', markersize=4)
    axes[1, 1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
    axes[1, 1].set_title('Learning Rate Schedule (Cosine Annealing)', fontsize=14, fontweight='bold')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(alpha=0.3, which='both')

    plt.suptitle('ResNet-50 Baseline Training History', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()

    # Save
    save_path = save_dir / 'training_curves_resnet50.png'
    plt.savefig(save_path, dpi=400, bbox_inches='tight')
    plt.show()

    print(f"✅ Training curves saved to: {save_path}")


# Plot training history
if len(history['train_loss']) > 0:
    plot_training_history(history, OUTPUT_DIR)

    # Print summary
    print("\n" + "="*60)
    print("📊 TRAINING SUMMARY")
    print("="*60)
    print(f"Total epochs: {len(history['train_loss'])}")
    print(f"Best epoch: {history['val_qwk'].index(max(history['val_qwk'])) + 1}")
    print(f"Best QWK: {max(history['val_qwk']):.4f}")
    print(f"Final train acc: {history['train_acc'][-1]:.4f}")
    print(f"Final val acc: {history['val_acc'][-1]:.4f}")
    print("="*60)
else:
    print("⚠️ No training history to plot. Run Cell 13 first!")

In [ ]:
# CELL 31: Load Best Model and Evaluate on Test Set (FIXED)

print("="*60)
print("🧪 TESTING BEST MODEL")
print("="*60)

# Load best model
checkpoint_path = checkpoint_dir / 'best_model.pth'

if checkpoint_path.exists():
    print(f"📂 Loading model from: {checkpoint_path}")

    # Fix: Add weights_only=False for compatibility
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    best_epoch = checkpoint['epoch']
    best_qwk = checkpoint['best_val_qwk']

    print(f"✅ Loaded model from epoch {best_epoch}")
    print(f"   Validation QWK: {best_qwk:.4f}")

    # Evaluate on test set
    print(f"\n🔍 Evaluating on test set...")
    test_loss, test_acc, test_qwk, test_precision, test_recall, test_f1, test_preds, test_labels = validate(
        model, test_loader, criterion, device
    )

    print("\n" + "="*60)
    print("📊 TEST SET RESULTS")
    print("="*60)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test QWK: {test_qwk:.4f}")

    print("\n📈 Per-Class Performance:")
    print("-"*60)
    print(f"{'Grade':<8} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
    print("-"*60)
    for i, (p, r, f) in enumerate(zip(test_precision, test_recall, test_f1)):
        print(f"{i:<8} {p:<12.4f} {r:<12.4f} {f:<12.4f}")
    print("-"*60)

    # Macro and weighted averages
    macro_f1 = np.mean(test_f1)
    weighted_f1 = np.average(test_f1, weights=np.bincount(test_labels))

    print(f"\nMacro-averaged F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print("="*60)

else:
    print(f"❌ Checkpoint not found at: {checkpoint_path}")
    print("   Run Cell 13 (training) first!")

In [ ]:
# CELL 32: Confusion Matrix and Analysis

def plot_confusion_matrix(y_true, y_pred, save_dir, title='Confusion Matrix'):
    """Plot confusion matrix heatmap"""
    cm = confusion_matrix(y_true, y_pred)

    # Normalize
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'Grade {i}' for i in range(5)],
                yticklabels=[f'Grade {i}' for i in range(5)],
                ax=axes[0], cbar_kws={'label': 'Count'},
                annot_kws={'size': 12})
    axes[0].set_xlabel('Predicted Label', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('True Label', fontsize=14, fontweight='bold')
    axes[0].set_title('Confusion Matrix (Counts)', fontsize=16, fontweight='bold')

    # Normalized
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='RdYlGn',
                xticklabels=[f'Grade {i}' for i in range(5)],
                yticklabels=[f'Grade {i}' for i in range(5)],
                ax=axes[1], cbar_kws={'label': 'Proportion'}, vmin=0, vmax=1,
                annot_kws={'size': 12})
    axes[1].set_xlabel('Predicted Label', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('True Label', fontsize=14, fontweight='bold')
    axes[1].set_title('Confusion Matrix (Normalized)', fontsize=16, fontweight='bold')

    plt.suptitle(f'{title} - ResNet-50 Baseline', fontsize=18, fontweight='bold')
    plt.tight_layout()

    # Save
    save_path = save_dir / f'confusion_matrix_{title.lower().replace(" ", "_")}.png'
    plt.savefig(save_path, dpi=400, bbox_inches='tight')
    plt.show()

    print(f"✅ Confusion matrix saved to: {save_path}")

    return cm


# Plot confusion matrix for test set
if 'test_preds' in locals():
    print("\n📊 Generating confusion matrix...\n")
    cm_test = plot_confusion_matrix(test_labels, test_preds, OUTPUT_DIR, title='Test Set')

    # Analysis
    print("\n" + "="*60)
    print("🔍 CONFUSION MATRIX ANALYSIS")
    print("="*60)

    # Diagonal (correct predictions)
    correct_per_class = np.diag(cm_test)
    total_per_class = cm_test.sum(axis=1)
    accuracy_per_class = correct_per_class / total_per_class

    print("\n📈 Per-Class Accuracy:")
    for i in range(5):
        print(f"   Grade {i}: {accuracy_per_class[i]:.2%} ({correct_per_class[i]}/{total_per_class[i]})")

    # Common misclassifications
    print("\n⚠️ Most Common Misclassifications:")
    misclass = []
    for i in range(5):
        for j in range(5):
            if i != j and cm_test[i, j] > 0:
                misclass.append((cm_test[i, j], i, j))

    misclass.sort(reverse=True)
    for count, true_label, pred_label in misclass[:5]:
        print(f"   Grade {true_label} → Grade {pred_label}: {count} cases")

    print("="*60)
else:
    print("⚠️ Test predictions not found")

In [ ]:
# CELL 33: Per-Class Performance Analysis

def plot_per_class_metrics(precision, recall, f1, labels, save_dir):
    """Plot per-class metrics comparison"""

    x = np.arange(len(labels))
    width = 0.25

    fig, ax = plt.subplots(figsize=(12, 6))

    bars1 = ax.bar(x - width, precision, width, label='Precision', color='#3498db', alpha=0.8)
    bars2 = ax.bar(x, recall, width, label='Recall', color='#e74c3c', alpha=0.8)
    bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='#2ecc71', alpha=0.8)

    ax.set_xlabel('Grade', fontsize=14, fontweight='bold')
    ax.set_ylabel('Score', fontsize=14, fontweight='bold')
    ax.set_title('Per-Class Performance Metrics - ResNet-50 Baseline', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'Grade {i}' for i in labels])
    ax.legend(fontsize=14)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.0])

    # Add value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}',
                   ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    save_path = save_dir / 'per_class_metrics.png'
    plt.savefig(save_path, dpi=400, bbox_inches='tight')
    plt.show()

    print(f"✅ Per-class metrics plot saved to: {save_path}")


# Plot per-class metrics
if 'test_precision' in locals():
    print("📊 Generating per-class metrics visualization...\n")
    plot_per_class_metrics(test_precision, test_recall, test_f1,
                           range(5), OUTPUT_DIR)

    # Detailed analysis table
    print("\n" + "="*80)
    print("📋 DETAILED PER-CLASS ANALYSIS")
    print("="*80)

    class_counts = np.bincount(test_labels)

    print(f"\n{'Grade':<8} {'Samples':<10} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-"*80)

    for i in range(5):
        print(f"{i:<8} {class_counts[i]:<10} {test_precision[i]:<12.4f} "
              f"{test_recall[i]:<12.4f} {test_f1[i]:<12.4f} {class_counts[i]:<10}")

    print("-"*80)
    print(f"{'Overall':<8} {len(test_labels):<10} {np.mean(test_precision):<12.4f} "
          f"{np.mean(test_recall):<12.4f} {np.mean(test_f1):<12.4f} {len(test_labels):<10}")
    print("="*80)

    # Identify best and worst performing classes
    best_f1_idx = np.argmax(test_f1)
    worst_f1_idx = np.argmin(test_f1)

    print(f"\n🏆 Best Performing Class: Grade {best_f1_idx} (F1 = {test_f1[best_f1_idx]:.4f})")
    print(f"⚠️ Worst Performing Class: Grade {worst_f1_idx} (F1 = {test_f1[worst_f1_idx]:.4f})")
    print(f"📊 Performance Gap: {test_f1[best_f1_idx] - test_f1[worst_f1_idx]:.4f}")

else:
    print("⚠️ Test metrics not found!")


In [ ]:
# CELL 34: ROC Curves for Multi-class Classification

def plot_multiclass_roc(model, loader, device, n_classes, save_dir):
    """Plot ROC curves for all classes"""

    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Computing ROC', leave=False):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # Convert to arrays
    y_true = np.array(all_labels)
    y_score = np.array(all_probs)

    # Binarize labels for multi-class ROC
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    # Compute ROC curve and AUC for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))

    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

    for i, color in enumerate(colors):
        ax.plot(fpr[i], tpr[i], color=color, lw=2,
               label=f'Grade {i} (AUC = {roc_auc[i]:.3f})')

    # Diagonal reference line
    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=14, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=14, fontweight='bold')
    ax.set_title('ROC Curves - ResNet-50 Baseline (One-vs-Rest)', fontsize=16, fontweight='bold')
    ax.legend(loc="lower right", fontsize=12)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    save_path = save_dir / 'roc_curves_multiclass.png'
    plt.savefig(save_path, dpi=400, bbox_inches='tight')
    plt.show()

    print(f"✅ ROC curves saved to: {save_path}")

    return roc_auc


# Generate ROC curves
if 'model' in locals() and 'test_loader' in locals():
    print("📈 Generating ROC curves...\n")
    roc_auc_scores = plot_multiclass_roc(model, test_loader, device, 5, OUTPUT_DIR)

    print("\n" + "="*60)
    print("📊 AUC SCORES (One-vs-Rest)")
    print("="*60)
    for grade, auc_score in roc_auc_scores.items():
        print(f"Grade {grade}: {auc_score:.4f}")

    mean_auc = np.mean(list(roc_auc_scores.values()))
    print(f"\nMean AUC: {mean_auc:.4f}")
    print("="*60)
else:
    print("⚠️ Model or test loader not found!")

In [ ]:
# CELL 35: Export Results Summary to CSV
def export_results_summary(history, test_metrics, save_dir):
    """Export all results to CSV files"""

    # 1. Training history
    history_df = pd.DataFrame({
        'Epoch': range(1, len(history['train_loss']) + 1),
        'Train_Loss': history['train_loss'],
        'Train_Accuracy': history['train_acc'],
        'Val_Loss': history['val_loss'],
        'Val_Accuracy': history['val_acc'],
        'Val_QWK': history['val_qwk'],
        'Learning_Rate': history['lr']
    })

    history_path = save_dir / 'training_history.csv'
    history_df.to_csv(history_path, index=False)
    print(f"✅ Training history saved to: {history_path}")

    # 2. Test results per class
    test_results_df = pd.DataFrame({
        'Grade': range(5),
        'Precision': test_metrics['precision'],
        'Recall': test_metrics['recall'],
        'F1_Score': test_metrics['f1'],
        'Support': np.bincount(test_metrics['labels'])
    })

    test_results_path = save_dir / 'test_results_per_class.csv'
    test_results_df.to_csv(test_results_path, index=False)
    print(f"✅ Test results saved to: {test_results_path}")

    # 3. Overall summary
    summary_data = {
        'Metric': [
            'Best_Epoch',
            'Best_Val_QWK',
            'Test_Accuracy',
            'Test_QWK',
            'Macro_F1',
            'Weighted_F1',
            'Total_Training_Time',
            'Total_Epochs',
            'Dataset_Size_Train',
            'Dataset_Size_Val',
            'Dataset_Size_Test'
        ],
        'Value': [
            history['val_qwk'].index(max(history['val_qwk'])) + 1,
            max(history['val_qwk']),
            test_metrics['accuracy'],
            test_metrics['qwk'],
            np.mean(test_metrics['f1']),
            np.average(test_metrics['f1'], weights=np.bincount(test_metrics['labels'])),
            f"{int(test_metrics['training_time'] // 3600)}h {int((test_metrics['training_time'] % 3600) // 60)}m",
            len(history['train_loss']),
            len(train_dataset_aug),
            len(val_dataset_aug),
            len(test_dataset_aug)
        ]
    }

    summary_df = pd.DataFrame(summary_data)
    summary_path = save_dir / 'model_summary.csv'
    summary_df.to_csv(summary_path, index=False)
    print(f"✅ Model summary saved to: {summary_path}")

    print("\n✅ All results exported successfully!")


# Prepare test metrics dictionary
if 'test_qwk' in locals():
    test_metrics_dict = {
        'precision': test_precision,
        'recall': test_recall,
        'f1': test_f1,
        'labels': test_labels,
        'accuracy': test_acc,
        'qwk': test_qwk,
        'training_time': 1 * 3600 + 23 * 60  # From Cell 13 output: 1h 23m
    }

    print("📁 Exporting results to CSV files...\n")
    export_results_summary(history, test_metrics_dict, OUTPUT_DIR)

else:
    print("⚠️ Test metrics not available. Run Cell 15 first!")

In [ ]:
# CELL 36: Generate Final Summary Report

def print_final_report(history, test_metrics):
    """Print comprehensive final report"""

    print("\n" + "="*80)
    print("📄 RESNET-50 BASELINE - FINAL REPORT")
    print("="*80)
    print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)

    # Dataset info
    print("\n📊 DATASET INFORMATION")
    print("-"*80)
    print(f"Training samples:   {len(train_dataset_aug):,}")
    print(f"Validation samples: {len(val_dataset_aug):,}")
    print(f"Test samples:       {len(test_dataset_aug):,}")
    print(f"Total samples:      {len(train_dataset_aug) + len(val_dataset_aug) + len(test_dataset_aug):,}")
    print(f"Classes:            5 (KL Grades 0-4)")
    print(f"Image size:         224×224 RGB")

    # Model info
    print("\n🔧 MODEL CONFIGURATION")
    print("-"*80)
    print(f"Architecture:       ResNet-50")
    print(f"Pretrained:         ImageNet")
    print(f"Optimizer:          AdamW")
    print(f"Learning rate:      1e-4 (Cosine Annealing)")
    print(f"Loss function:      Weighted Cross-Entropy")
    print(f"Batch size:         32")
    print(f"Device:             {device}")

    # Training info
    print("\n⏱️ TRAINING SUMMARY")
    print("-"*80)
    best_epoch = history['val_qwk'].index(max(history['val_qwk'])) + 1
    print(f"Total epochs:       {len(history['train_loss'])}")
    print(f"Best epoch:         {best_epoch}")
    print(f"Training time:      1h 23m")
    print(f"Early stopping:     Triggered at epoch 24 (patience 15)")

    # Performance metrics
    print("\n📈 VALIDATION PERFORMANCE (Best Epoch)")
    print("-"*80)
    print(f"Best Val QWK:       {max(history['val_qwk']):.4f}")
    print(f"Val Accuracy:       {history['val_acc'][best_epoch-1]:.4f}")
    print(f"Val Loss:           {history['val_loss'][best_epoch-1]:.4f}")

    print("\n📈 TEST SET PERFORMANCE")
    print("-"*80)
    print(f"Test QWK:           {test_metrics['qwk']:.4f} ⭐")
    print(f"Test Accuracy:      {test_metrics['accuracy']:.4f}")
    print(f"Test Loss:          {test_loss:.4f}")
    print(f"Macro-avg F1:       {np.mean(test_metrics['f1']):.4f}")
    print(f"Weighted F1:        {np.average(test_metrics['f1'], weights=np.bincount(test_metrics['labels'])):.4f}")

    # Per-class breakdown
    print("\n📋 PER-CLASS PERFORMANCE (Test Set)")
    print("-"*80)
    print(f"{'Grade':<10} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
    print("-"*80)
    for i in range(5):
        print(f"{i:<10} {test_metrics['precision'][i]:<12.4f} "
              f"{test_metrics['recall'][i]:<12.4f} {test_metrics['f1'][i]:<12.4f}")



    print("\n" + "="*80)
    print("✅ BASELINE MODEL TRAINING COMPLETE")
    print("="*80)

    # Save report to file
    report_path = OUTPUT_DIR / 'final_report.txt'
    with open(report_path, 'w') as f:
        # Redirect print to file (simplified version)
        f.write("="*80 + "\n")
        f.write("RESNET-50 BASELINE - FINAL REPORT\n")
        f.write("="*80 + "\n")
        f.write(f"Test QWK: {test_metrics['qwk']:.4f}\n")
        f.write(f"Test Accuracy: {test_metrics['accuracy']:.4f}\n")
        f.write(f"Macro F1: {np.mean(test_metrics['f1']):.4f}\n")

    print(f"\n💾 Report saved to: {report_path}")


# Generate final report
if 'test_metrics_dict' in locals():
    print_final_report(history, test_metrics_dict)
else:
    print("⚠️ Complete test metrics not available!")

In [ ]:
# CELL 37: Vision Transformer Branch for Hybrid Model
class VisionTransformerBranch(nn.Module):
    """
    Vision Transformer branch for global feature extraction
    """
    def __init__(self, model_name='vit_base_patch16_224', pretrained=True, num_classes=5):
        super(VisionTransformerBranch, self).__init__()

        # Load pretrained ViT
        self.vit = timm.create_model(model_name, pretrained=pretrained, num_classes=0)

        # Get feature dimension
        self.feature_dim = self.vit.num_features

        print(f"✅ ViT Branch initialized:")
        print(f"   Model: {model_name}")
        print(f"   Feature dim: {self.feature_dim}")
        print(f"   Pretrained: {pretrained}")

    def forward(self, x):
        """
        Args:
            x: Input tensor [B, 3, 224, 224]
        Returns:
            features: [B, feature_dim] (typically 768 for ViT-Base)
        """
        features = self.vit(x)
        return features


# Test ViT branch
print("🔧 Initializing Vision Transformer branch...\n")

vit_branch = VisionTransformerBranch(
    model_name='vit_base_patch16_224',
    pretrained=True
)
vit_branch = vit_branch.to(device)

# Test forward pass
print("\n🧪 Testing ViT forward pass...")
dummy_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    vit_features = vit_branch(dummy_input)

print(f"   Input shape: {dummy_input.shape}")
print(f"   Output shape: {vit_features.shape}")
print(f"   Feature dimension: {vit_features.shape[1]}")

print("\n✅ ViT branch ready!")

In [ ]:
# CELL 38: CNN Branch for Local Feature Extraction

class CNNBranch(nn.Module):
    """
    ResNet-50 branch for local feature extraction
    """
    def __init__(self, pretrained=True):
        super(CNNBranch, self).__init__()

        # Load pretrained ResNet-50
        resnet = models.resnet50(pretrained=pretrained)

        # Remove final FC layer (keep feature extractor only)
        self.features = nn.Sequential(*list(resnet.children())[:-1])

        # Feature dimension
        self.feature_dim = 2048

        print(f"✅ CNN Branch initialized:")
        print(f"   Model: ResNet-50")
        print(f"   Feature dim: {self.feature_dim}")
        print(f"   Pretrained: {pretrained}")

    def forward(self, x):
        """
        Args:
            x: Input tensor [B, 3, 224, 224]
        Returns:
            features: [B, 2048]
        """
        features = self.features(x)
        features = features.view(features.size(0), -1)  # Flatten
        return features


# Test CNN branch
print("🔧 Initializing CNN branch...\n")

cnn_branch = CNNBranch(pretrained=True)
cnn_branch = cnn_branch.to(device)

# Test forward pass
print("\n🧪 Testing CNN forward pass...")
with torch.no_grad():
    cnn_features = cnn_branch(dummy_input)

print(f"   Input shape: {dummy_input.shape}")
print(f"   Output shape: {cnn_features.shape}")
print(f"   Feature dimension: {cnn_features.shape[1]}")

print("\n✅ CNN branch ready!")

In [ ]:
# CELL 39: Cross-Attention Fusion Module

class CrossAttentionFusion(nn.Module):
    """
    Cross-attention mechanism to fuse CNN and ViT features
    """
    def __init__(self, cnn_dim, vit_dim, hidden_dim=512, num_heads=8):
        super(CrossAttentionFusion, self).__init__()

        self.cnn_dim = cnn_dim
        self.vit_dim = vit_dim
        self.hidden_dim = hidden_dim

        # Project features to same dimension
        self.cnn_proj = nn.Linear(cnn_dim, hidden_dim)
        self.vit_proj = nn.Linear(vit_dim, hidden_dim)

        # Multi-head cross-attention
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )

        # Layer normalization
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(0.1)
        )

        print(f"✅ Cross-Attention Fusion initialized:")
        print(f"   CNN dim: {cnn_dim} → {hidden_dim}")
        print(f"   ViT dim: {vit_dim} → {hidden_dim}")
        print(f"   Num heads: {num_heads}")

    def forward(self, cnn_feat, vit_feat):
        """
        Args:
            cnn_feat: [B, cnn_dim]
            vit_feat: [B, vit_dim]
        Returns:
            fused_feat: [B, hidden_dim]
        """
        # Project to same dimension
        cnn_proj = self.cnn_proj(cnn_feat)  # [B, hidden_dim]
        vit_proj = self.vit_proj(vit_feat)  # [B, hidden_dim]

        # Add sequence dimension for attention
        cnn_proj = cnn_proj.unsqueeze(1)  # [B, 1, hidden_dim]
        vit_proj = vit_proj.unsqueeze(1)  # [B, 1, hidden_dim]

        # Cross-attention: CNN attends to ViT
        attn_out, attn_weights = self.cross_attn(
            query=cnn_proj,
            key=vit_proj,
            value=vit_proj
        )

        # Residual connection + norm
        fused = self.norm1(cnn_proj + attn_out)

        # Feed-forward network
        ffn_out = self.ffn(fused)
        fused = self.norm2(fused + ffn_out)

        # Remove sequence dimension
        fused = fused.squeeze(1)  # [B, hidden_dim]

        return fused, attn_weights


# Test cross-attention fusion
print("🔧 Initializing Cross-Attention Fusion...\n")

fusion_module = CrossAttentionFusion(
    cnn_dim=2048,
    vit_dim=768,
    hidden_dim=512,
    num_heads=8
)
fusion_module = fusion_module.to(device)

# Test forward pass
print("\n🧪 Testing fusion forward pass...")
with torch.no_grad():
    fused_features, attention_weights = fusion_module(cnn_features, vit_features)

print(f"   CNN features: {cnn_features.shape}")
print(f"   ViT features: {vit_features.shape}")
print(f"   Fused features: {fused_features.shape}")
print(f"   Attention weights: {attention_weights.shape}")

print("\n✅ Cross-attention fusion ready!")

In [ ]:
# CELL 40: Hybrid CNN-Transformer Model

class HybridCNNTransformer(nn.Module):
    """
    Hybrid CNN-Transformer for Knee OA Classification
    Combines ResNet-50 (local) + ViT (global) with cross-attention fusion
    """
    def __init__(self, num_classes=5, hidden_dim=512, num_heads=8, dropout=0.5):
        super(HybridCNNTransformer, self).__init__()

        # Branches
        self.cnn_branch = CNNBranch(pretrained=True)
        self.vit_branch = VisionTransformerBranch(
            model_name='vit_base_patch16_224',
            pretrained=True
        )

        # Fusion
        self.fusion = CrossAttentionFusion(
            cnn_dim=self.cnn_branch.feature_dim,
            vit_dim=self.vit_branch.feature_dim,
            hidden_dim=hidden_dim,
            num_heads=num_heads
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

        print(f"\n{'='*60}")
        print("🎯 HYBRID CNN-TRANSFORMER MODEL")
        print(f"{'='*60}")
        print(f"CNN Branch: ResNet-50 → {self.cnn_branch.feature_dim}D")
        print(f"ViT Branch: ViT-Base → {self.vit_branch.feature_dim}D")
        print(f"Fusion: Cross-Attention → {hidden_dim}D")
        print(f"Classifier: {hidden_dim} → {num_classes} classes")
        print(f"{'='*60}")

    def forward(self, x):
        """
        Args:
            x: Input images [B, 3, 224, 224]
        Returns:
            logits: [B, num_classes]
            attention_weights: [B, num_heads, 1, 1] (for visualization)
        """
        # Extract features from both branches
        cnn_features = self.cnn_branch(x)
        vit_features = self.vit_branch(x)

        # Fuse features with cross-attention
        fused_features, attention_weights = self.fusion(cnn_features, vit_features)

        # Classification
        logits = self.classifier(fused_features)

        return logits, attention_weights


# Initialize hybrid model
print("🔧 Initializing Hybrid CNN-Transformer model...\n")

hybrid_model = HybridCNNTransformer(
    num_classes=5,
    hidden_dim=512,
    num_heads=8,
    dropout=0.5
)
hybrid_model = hybrid_model.to(device)

# Count parameters
total_params = sum(p.numel() for p in hybrid_model.parameters())
trainable_params = sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)

print(f"\n📊 Model Statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

# Test forward pass
print(f"\n🧪 Testing hybrid model forward pass...")
dummy_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    logits, attn_weights = hybrid_model(dummy_input)

print(f"   Input shape: {dummy_input.shape}")
print(f"   Output logits: {logits.shape}")
print(f"   Attention weights: {attn_weights.shape}")

print(f"\n✅ Hybrid model ready for training!")

In [ ]:
# CELL 41: Train Hybrid CNN-Transformer Model # Training configuration

NUM_EPOCHS_HYBRID = 30
PATIENCE_HYBRID = 15

# Setup optimizer and scheduler for hybrid model
optimizer_hybrid = optim.AdamW(
    hybrid_model.parameters(),
    lr=5e-5,
    weight_decay=0.01
)

scheduler_hybrid = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_hybrid,
    T_max=NUM_EPOCHS_HYBRID,
    eta_min=1e-6
)

# Loss function (same weighted cross-entropy)
criterion_hybrid = nn.CrossEntropyLoss(weight=class_weights).to(device)

# Training history
history_hybrid = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_qwk': [],
    'lr': []
}

# Best model tracking
best_val_qwk_hybrid = 0.0
best_epoch_hybrid = 0
patience_counter_hybrid = 0

# Checkpoint directory
checkpoint_dir_hybrid = OUTPUT_DIR / 'checkpoints_hybrid'
checkpoint_dir_hybrid.mkdir(parents=True, exist_ok=True)

print("="*60)
print("🚀 STARTING HYBRID MODEL TRAINING")
print("="*60)
print(f"Model: Hybrid CNN-Transformer")
print(f"Device: {device}")
print(f"Parameters: {trainable_params:,}")
print(f"Training samples: {len(train_dataset_aug)}")
print(f"Validation samples: {len(val_dataset_aug)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {NUM_EPOCHS_HYBRID}")
print(f"Learning rate: 5e-5")
print(f"Early stopping patience: {PATIENCE_HYBRID}")
print(f"Seed: 456")  # ← CHANGED: Show seed
print("="*60)

# Modified training function for hybrid model
def train_hybrid_one_epoch(model, loader, criterion, optimizer, device):
    """Train hybrid model for one epoch"""
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        logits, _ = model(images)
        loss = criterion(logits, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(logits, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc


# Modified validation function for hybrid model
def validate_hybrid(model, loader, criterion, device):
    """Validate hybrid model"""
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation', leave=False):
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            logits, _ = model(images)
            loss = criterion(logits, labels)

            # Statistics
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(logits, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    # Per-class metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )

    # QWK
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

    return epoch_loss, epoch_acc, qwk, precision, recall, f1, all_preds, all_labels


print("\n✅ Training functions defined for hybrid model!")

# Start training
start_time = time.time()

for epoch in range(NUM_EPOCHS_HYBRID):
    epoch_start = time.time()

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS_HYBRID}")
    print(f"{'='*60}")

    # Get current learning rate
    current_lr = optimizer_hybrid.param_groups[0]['lr']
    print(f"Learning Rate: {current_lr:.6f}")

    # Train
    train_loss, train_acc = train_hybrid_one_epoch(
        hybrid_model, train_loader, criterion_hybrid, optimizer_hybrid, device
    )

    # Validate
    val_loss, val_acc, val_qwk, precision, recall, f1, _, _ = validate_hybrid(
        hybrid_model, val_loader, criterion_hybrid, device
    )

    # Update scheduler
    scheduler_hybrid.step()

    # Save history
    history_hybrid['train_loss'].append(train_loss)
    history_hybrid['train_acc'].append(train_acc)
    history_hybrid['val_loss'].append(val_loss)
    history_hybrid['val_acc'].append(val_acc)
    history_hybrid['val_qwk'].append(val_qwk)
    history_hybrid['lr'].append(current_lr)

    # Print metrics
    epoch_time = time.time() - epoch_start
    print(f"\n📊 Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"   Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"   Val QWK:    {val_qwk:.4f}")
    print(f"\n📈 Per-Class F1-Scores:")
    for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
        print(f"   Grade {i}: P={p:.3f} R={r:.3f} F1={f:.3f}")
    print(f"\n⏱️ Epoch time: {epoch_time:.1f}s")

    # Save best model
    if val_qwk > best_val_qwk_hybrid:
        best_val_qwk_hybrid = val_qwk
        best_epoch_hybrid = epoch + 1
        patience_counter_hybrid = 0

        # ← CHANGED: Save with seed in filename
        checkpoint_path_hybrid = checkpoint_dir_hybrid / 'best_hybrid_model_seed456.pth'

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': hybrid_model.state_dict(),
            'optimizer_state_dict': optimizer_hybrid.state_dict(),
            'scheduler_state_dict': scheduler_hybrid.state_dict(),
            'best_val_qwk': best_val_qwk_hybrid,
            'history': history_hybrid,
            'seed': 456  # ← CHANGED: Add seed
        }, checkpoint_path_hybrid)

        print(f"\n💾 Best hybrid model saved! (QWK: {best_val_qwk_hybrid:.4f})")
    else:
        patience_counter_hybrid += 1
        print(f"\n⏸️ No improvement. Patience: {patience_counter_hybrid}/{PATIENCE_HYBRID}")

    # Early stopping
    if patience_counter_hybrid >= PATIENCE_HYBRID:
        print(f"\n🛑 Early stopping triggered at epoch {epoch+1}")
        print(f"   Best QWK: {best_val_qwk_hybrid:.4f} at epoch {best_epoch_hybrid}")
        break

# Training complete
total_time = time.time() - start_time
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)

print("\n" + "="*60)
print("✅ HYBRID MODEL TRAINING COMPLETE!")
print("="*60)
print(f"Total time: {hours}h {minutes}m")
print(f"Best validation QWK: {best_val_qwk_hybrid:.4f} (epoch {best_epoch_hybrid})")
print(f"Baseline QWK: 0.7971")
print(f"Improvement: +{best_val_qwk_hybrid - 0.7971:.4f}")
print(f"Best model saved at: {checkpoint_path_hybrid}")
print(f"Filename: best_hybrid_model_seed456.pth")  # ← CHANGED: Show filename
print("="*60)

In [ ]:
# CELL 42 Dataset class
class KneeOADataset(Dataset):
    def __init__(self, data_dir, split='train', transform=None):
        self.data_dir = Path(data_dir) / split
        self.transform = transform
        self.images = []
        self.labels = []

        grade_folders = sorted([f for f in self.data_dir.iterdir() if f.is_dir()])

        for idx, grade_folder in enumerate(grade_folders):
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
                for img_path in grade_folder.glob(ext):
                    self.images.append(str(img_path))
                    self.labels.append(idx)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, label

# Transforms
test_transform = A.Compose([
    A.Resize(224, 224),
    A.CLAHE(clip_limit=2.0, p=1.0),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Load test dataset
test_dataset = KneeOADataset(DATA_DIR, split='test', transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"\n✅ Test dataset loaded: {len(test_dataset)} images")

In [ ]:
# CELL 43 Model architectures
class ResNet50Baseline(nn.Module):
    def __init__(self, num_classes=5, pretrained=True):
        super(ResNet50Baseline, self).__init__()
        self.backbone = models.resnet50(pretrained=pretrained)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)


In [ ]:
# CELL 44
class VisionTransformerBranch(nn.Module):
    def __init__(self, model_name='vit_base_patch16_224', pretrained=True):
        super(VisionTransformerBranch, self).__init__()
        self.vit = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.feature_dim = self.vit.num_features

    def forward(self, x):
        return self.vit(x)


class CNNBranch(nn.Module):
    def __init__(self, pretrained=True):
        super(CNNBranch, self).__init__()
        resnet = models.resnet50(pretrained=pretrained)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_dim = 2048

    def forward(self, x):
        features = self.features(x)
        return features.view(features.size(0), -1)


class CrossAttentionFusion(nn.Module):
    def __init__(self, cnn_dim, vit_dim, hidden_dim=512, num_heads=8):
        super(CrossAttentionFusion, self).__init__()
        self.cnn_proj = nn.Linear(cnn_dim, hidden_dim)
        self.vit_proj = nn.Linear(vit_dim, hidden_dim)
        self.cross_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=0.1, batch_first=True)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(0.1)
        )

    def forward(self, cnn_feat, vit_feat):
        cnn_proj = self.cnn_proj(cnn_feat).unsqueeze(1)
        vit_proj = self.vit_proj(vit_feat).unsqueeze(1)
        attn_out, attn_weights = self.cross_attn(cnn_proj, vit_proj, vit_proj)
        fused = self.norm1(cnn_proj + attn_out)
        ffn_out = self.ffn(fused)
        fused = self.norm2(fused + ffn_out)
        return fused.squeeze(1), attn_weights


class HybridCNNTransformer(nn.Module):
    def __init__(self, num_classes=5, hidden_dim=512, num_heads=8, dropout=0.5):
        super(HybridCNNTransformer, self).__init__()
        self.cnn_branch = CNNBranch(pretrained=True)
        self.vit_branch = VisionTransformerBranch('vit_base_patch16_224', pretrained=True)
        self.fusion = CrossAttentionFusion(
            self.cnn_branch.feature_dim,
            self.vit_branch.feature_dim,
            hidden_dim, num_heads
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        cnn_features = self.cnn_branch(x)
        vit_features = self.vit_branch(x)
        fused_features, attention_weights = self.fusion(cnn_features, vit_features)
        logits = self.classifier(fused_features)
        return logits, attention_weights


# Load baseline model
print("\n📂 Loading baseline model...")
baseline_model = ResNet50Baseline(num_classes=5, pretrained=False)
baseline_checkpoint = torch.load(
    OUTPUT_DIR / 'checkpoints' / 'best_model.pth',
    map_location=device,
    weights_only=False
)
baseline_model.load_state_dict(baseline_checkpoint['model_state_dict'])
baseline_model = baseline_model.to(device)
baseline_model.eval()
print("✅ Baseline model loaded")

# Load hybrid model
print("\n📂 Loading hybrid model...")
hybrid_model = HybridCNNTransformer(num_classes=5, hidden_dim=512, num_heads=8, dropout=0.5)
hybrid_checkpoint = torch.load(
    OUTPUT_DIR / 'checkpoints_hybrid' / 'best_hybrid_model.pth',
    map_location=device,
    weights_only=False
)
hybrid_model.load_state_dict(hybrid_checkpoint['model_state_dict'])
hybrid_model = hybrid_model.to(device)
hybrid_model.eval()
print("✅ Hybrid model loaded")

print("\n" + "="*60)
print("✅ SETUP COMPLETE - READY FOR EVALUATION")
print("="*60)

In [ ]:
# CELL 45 Load models
baseline_model = ResNet50Baseline(num_classes=5, pretrained=False).to(device)
checkpoint = torch.load(OUTPUT_DIR / 'checkpoints' / 'best_model.pth',
                        map_location=device, weights_only=False)
baseline_model.load_state_dict(checkpoint['model_state_dict'])
baseline_model.eval()
print("✅ Baseline loaded")

hybrid_model = HybridCNNTransformer(num_classes=5).to(device)
checkpoint = torch.load(OUTPUT_DIR / 'checkpoints_hybrid' / 'best_hybrid_model.pth',
                        map_location=device, weights_only=False)
hybrid_model.load_state_dict(checkpoint['model_state_dict'])
hybrid_model.eval()
print("✅ Hybrid loaded")

In [ ]:
# CELL 46: IoU Expansion (150+ samples) + Inference Latency
print("="*60)
print("📊 IoU EXPANSION + INFERENCE LATENCY ANALYSIS")
print("="*60)

# ============================================================
# PART 1: IoU EXPANSION
# ============================================================

class AttentionRollout:
    def __init__(self, model, device, head_fusion='mean', discard_ratio=0.9):
        self.model = model
        self.device = device
        self.head_fusion = head_fusion
        self.discard_ratio = discard_ratio

    def get_attention_maps(self, input_tensor):
        self.model.eval()
        vit = self.model.vit_branch.vit

        with torch.no_grad():
            x = vit.patch_embed(input_tensor)
            if hasattr(vit, 'cls_token'):
                cls_token = vit.cls_token.expand(x.shape[0], -1, -1)
                x = torch.cat((cls_token, x), dim=1)
            x = x + vit.pos_embed
            x = vit.pos_drop(x)

            attentions = []
            for blk in vit.blocks:
                B, N, C = x.shape
                qkv = blk.attn.qkv(x).reshape(B, N, 3, blk.attn.num_heads,
                       C // blk.attn.num_heads).permute(2, 0, 3, 1, 4)
                q, k, v = qkv[0], qkv[1], qkv[2]
                attn = (q @ k.transpose(-2, -1)) * blk.attn.scale
                attn = attn.softmax(dim=-1)
                attentions.append(attn.mean(dim=1))
                x = blk(x)

            rollout = self.compute_rollout(attentions)
        return rollout

    def compute_rollout(self, attentions):
        result = torch.eye(attentions[0].size(-1)).to(self.device)
        for attention in attentions:
            attention_heads_fused = attention.mean(dim=0)
            attention_heads_fused += torch.eye(attention_heads_fused.size(-1)).to(self.device)
            attention_heads_fused /= attention_heads_fused.sum(dim=-1, keepdim=True)
            result = torch.matmul(attention_heads_fused, result)

        mask = result[0, 1:]
        width = int(mask.size(-1) ** 0.5)
        mask = mask.reshape(width, width).cpu().numpy()
        mask = cv2.resize(mask, (224, 224))
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        return mask


def calculate_iou(mask1, mask2, threshold=0.5):
    binary1 = (mask1 > threshold).astype(np.float32)
    binary2 = (mask2 > threshold).astype(np.float32)
    intersection = np.logical_and(binary1, binary2).sum()
    union = np.logical_or(binary1, binary2).sum()
    if union == 0:
        return 0.0
    return intersection / union


# Setup GradCAM
target_layers = [hybrid_model.cnn_branch.features[-2][-1]]
grad_cam = GradCAM(model=hybrid_model, target_layers=target_layers)
attention_rollout = AttentionRollout(hybrid_model, device)

# Collect 150 samples from test set
print("\n🔄 Collecting 150 test samples...")
all_images = []
all_labels = []

for images, labels in test_loader:
    for i in range(len(images)):
        all_images.append(images[i])
        all_labels.append(labels[i].item())
    if len(all_images) >= 150:
        break

all_images = all_images[:150]
all_labels = all_labels[:150]
print(f"✅ Collected {len(all_images)} samples")

# Calculate IoU for all 150 samples
print("\n🔄 Calculating IoU for 150 samples...")
iou_scores = []
hybrid_model.eval()

for idx in range(len(all_images)):
    input_tensor = all_images[idx].unsqueeze(0).to(device)
    label = all_labels[idx]

    try:
        # Grad-CAM
        targets = [ClassifierOutputTarget(label)]
        gradcam_map = grad_cam(input_tensor=input_tensor, targets=targets)
        gradcam_map = gradcam_map[0]
        gradcam_norm = (gradcam_map - gradcam_map.min()) / (gradcam_map.max() - gradcam_map.min() + 1e-8)

        # Attention Rollout
        attn_map = attention_rollout.get_attention_maps(input_tensor)

        # IoU
        iou = calculate_iou(gradcam_norm, attn_map)
        iou_scores.append(iou)

    except Exception as e:
        continue

    if (idx + 1) % 30 == 0:
        print(f"   Processed {idx+1}/150 samples...")

iou_scores = np.array(iou_scores)

print(f"\n{'='*60}")
print(f"📊 IoU RESULTS (150 samples)")
print(f"{'='*60}")
print(f"Mean IoU:    {iou_scores.mean():.4f}")
print(f"Std IoU:     {iou_scores.std():.4f}")
print(f"Min IoU:     {iou_scores.min():.4f}")
print(f"Max IoU:     {iou_scores.max():.4f}")
print(f"Median IoU:  {np.median(iou_scores):.4f}")
print(f"Samples:     {len(iou_scores)}")

# Save IoU results
iou_df = pd.DataFrame({
    'Sample': range(1, len(iou_scores)+1),
    'IoU_Score': iou_scores,
    'Grade': all_labels[:len(iou_scores)]
})
iou_path = OUTPUT_DIR / 'new' / 'IoU_150_samples.csv'
iou_df.to_csv(iou_path, index=False)
print(f"\n✅ IoU results saved: {iou_path}")

# ============================================================
# PART 2: INFERENCE LATENCY
# ============================================================

print(f"\n{'='*60}")
print("⏱️ INFERENCE LATENCY ANALYSIS")
print(f"{'='*60}")

# Test single batch
test_batch = next(iter(test_loader))[0][:32].to(device)

def measure_latency(model, input_batch, n_runs=50, is_hybrid=False):
    model.eval()
    times = []
    with torch.no_grad():
        # Warmup
        for _ in range(5):
            if is_hybrid:
                _ = model(input_batch)
            else:
                _ = model(input_batch)

        # Measure
        for _ in range(n_runs):
            start = time.time()
            if is_hybrid:
                _ = model(input_batch)
            else:
                _ = model(input_batch)
            times.append(time.time() - start)

    times = np.array(times)
    return {
        'mean_ms': times.mean() * 1000,
        'std_ms': times.std() * 1000,
        'per_image_ms': (times.mean() / input_batch.shape[0]) * 1000
    }

print("\n🔄 Measuring baseline latency...")
baseline_lat = measure_latency(baseline_model, test_batch, is_hybrid=False)

print("🔄 Measuring hybrid latency...")
hybrid_lat = measure_latency(hybrid_model, test_batch, is_hybrid=True)

print(f"\n{'='*60}")
print(f"📊 LATENCY RESULTS (batch_size=32, {50} runs)")
print(f"{'='*60}")
print(f"\nResNet-50 Baseline:")
print(f"  Batch latency:     {baseline_lat['mean_ms']:.2f} ± {baseline_lat['std_ms']:.2f} ms")
print(f"  Per-image latency: {baseline_lat['per_image_ms']:.2f} ms")

print(f"\nHybrid CNN-Transformer:")
print(f"  Batch latency:     {hybrid_lat['mean_ms']:.2f} ± {hybrid_lat['std_ms']:.2f} ms")
print(f"  Per-image latency: {hybrid_lat['per_image_ms']:.2f} ms")

print(f"\nRatio: {hybrid_lat['mean_ms']/baseline_lat['mean_ms']:.2f}x slower")

# Save latency results
latency_df = pd.DataFrame([
    {
        'Model': 'ResNet-50 Baseline',
        'Batch_Latency_ms': f"{baseline_lat['mean_ms']:.2f} ± {baseline_lat['std_ms']:.2f}",
        'Per_Image_ms': f"{baseline_lat['per_image_ms']:.2f}",
        'Parameters_M': 24.6
    },
    {
        'Model': 'Hybrid CNN-Transformer',
        'Batch_Latency_ms': f"{hybrid_lat['mean_ms']:.2f} ± {hybrid_lat['std_ms']:.2f}",
        'Per_Image_ms': f"{hybrid_lat['per_image_ms']:.2f}",
        'Parameters_M': 114.0
    }
])

latency_path = OUTPUT_DIR / 'new' / 'inference_latency.csv'
latency_df.to_csv(latency_path, index=False)
print(f"\n✅ Latency results saved: {latency_path}")

print(f"\n{'='*60}")
print("✅ ALL ANALYSIS COMPLETE!")
print(f"{'='*60}")

In [ ]:
# CELL 47: Test Set Evaluation

def evaluate_model(model, loader, device, is_hybrid=False):
    """Evaluate model on test set"""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images = images.to(device)

            if is_hybrid:
                outputs, _ = model(images)
            else:
                outputs = model(images)

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average=None, zero_division=0
    )

    return {
        'predictions': all_preds,
        'labels': all_labels,
        'accuracy': accuracy,
        'qwk': qwk,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


print("="*60)
print("🧪 EVALUATING MODELS ON TEST SET")
print("="*60)

# Evaluate baseline
print("\n📊 Evaluating Baseline Model...")
baseline_results = evaluate_model(baseline_model, test_loader, device, is_hybrid=False)

print("\n✅ Baseline Results:")
print(f"   Accuracy: {baseline_results['accuracy']:.4f}")
print(f"   QWK: {baseline_results['qwk']:.4f}")

# Evaluate hybrid
print("\n📊 Evaluating Hybrid Model...")
hybrid_results = evaluate_model(hybrid_model, test_loader, device, is_hybrid=True)

print("\n✅ Hybrid Results:")
print(f"   Accuracy: {hybrid_results['accuracy']:.4f}")
print(f"   QWK: {hybrid_results['qwk']:.4f}")

# Comparison
print("\n" + "="*60)
print("📊 COMPARISON")
print("="*60)
print(f"{'Metric':<20} {'Baseline':<15} {'Hybrid':<15} {'Improvement'}")
print("-"*60)
print(f"{'Accuracy':<20} {baseline_results['accuracy']:<15.4f} {hybrid_results['accuracy']:<15.4f} {(hybrid_results['accuracy']-baseline_results['accuracy'])*100:+.2f}%")
print(f"{'QWK':<20} {baseline_results['qwk']:<15.4f} {hybrid_results['qwk']:<15.4f} {(hybrid_results['qwk']-baseline_results['qwk'])*100:+.2f}%")

print("\n📈 Per-Class F1-Scores:")
print(f"{'Grade':<10} {'Baseline':<15} {'Hybrid':<15} {'Improvement'}")
print("-"*60)
for i in range(5):
    improvement = (hybrid_results['f1'][i] - baseline_results['f1'][i]) * 100
    print(f"{i:<10} {baseline_results['f1'][i]:<15.4f} {hybrid_results['f1'][i]:<15.4f} {improvement:+.2f}%")

print("="*60)

In [ ]:
# CELL 48 Bridge cell - extract variables for Cell 32, 33, 34
test_preds = baseline_results['predictions']
test_labels = baseline_results['labels']
test_precision = baseline_results['precision']
test_recall = baseline_results['recall']
test_f1 = baseline_results['f1']
test_acc = baseline_results['accuracy']
test_qwk = baseline_results['qwk']
model = baseline_model
print("✅ Variables ready for visualization cells")

In [ ]:
# CELL 49: Generate All Visualizations

# Confusion matrices comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cm_baseline = confusion_matrix(baseline_results['labels'], baseline_results['predictions'])
cm_hybrid = confusion_matrix(hybrid_results['labels'], hybrid_results['predictions'])

# Normalize
cm_baseline_norm = cm_baseline.astype('float') / cm_baseline.sum(axis=1)[:, np.newaxis]
cm_hybrid_norm = cm_hybrid.astype('float') / cm_hybrid.sum(axis=1)[:, np.newaxis]

# Baseline
sns.heatmap(cm_baseline_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[f'G{i}' for i in range(5)],
            yticklabels=[f'G{i}' for i in range(5)],
            ax=axes[0], vmin=0, vmax=1)
axes[0].set_title('Baseline ResNet-50', fontsize=17, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=16)
axes[0].set_ylabel('True', fontsize=16)

# Hybrid
sns.heatmap(cm_hybrid_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=[f'G{i}' for i in range(5)],
            yticklabels=[f'G{i}' for i in range(5)],
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Hybrid CNN-Transformer', fontsize=15, fontweight='bold')
axes[1].set_xlabel('Predicted', fontsize=14)
axes[1].set_ylabel('True', fontsize=14)

plt.suptitle('Confusion Matrix Comparison (Normalized)', fontsize=19, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_comparison.png', dpi=400, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix saved")

# Performance comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Overall metrics
metrics = ['QWK', 'Accuracy', 'Macro F1']
baseline_vals = [
    baseline_results['qwk'],
    baseline_results['accuracy'],
    np.mean(baseline_results['f1'])
]
hybrid_vals = [
    hybrid_results['qwk'],
    hybrid_results['accuracy'],
    np.mean(hybrid_results['f1'])
]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, baseline_vals, width, label='Baseline', color='#3498db', alpha=0.8)
axes[0].bar(x + width/2, hybrid_vals, width, label='Hybrid', color='#2ecc71', alpha=0.8)
axes[0].set_ylabel('Score', fontsize=11, fontweight='bold')
axes[0].set_title('Overall Metrics', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0.5, 1.0])

# Per-class F1
x = np.arange(5)
axes[1].bar(x - width/2, baseline_results['f1'], width, label='Baseline', color='#3498db', alpha=0.8)
axes[1].bar(x + width/2, hybrid_results['f1'], width, label='Hybrid', color='#2ecc71', alpha=0.8)
axes[1].set_xlabel('Grade', fontsize=16, fontweight='bold')
axes[1].set_ylabel('F1-Score', fontsize=16, fontweight='bold')
axes[1].set_title('Per-Class F1-Score', fontsize=15, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Grade {i}' for i in range(5)])
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'performance_comparison.png', dpi=400, bbox_inches='tight')
plt.show()
print("✅ Performance comparison saved")

In [ ]:
# CELL 50: Export All Results

# Model comparison
comparison_df = pd.DataFrame({
    'Model': ['Baseline ResNet-50', 'Hybrid CNN-Transformer'],
    'Test_QWK': [baseline_results['qwk'], hybrid_results['qwk']],
    'Test_Accuracy': [baseline_results['accuracy'], hybrid_results['accuracy']],
    'Macro_F1': [np.mean(baseline_results['f1']), np.mean(hybrid_results['f1'])],
    'Grade_0_F1': [baseline_results['f1'][0], hybrid_results['f1'][0]],
    'Grade_1_F1': [baseline_results['f1'][1], hybrid_results['f1'][1]],
    'Grade_2_F1': [baseline_results['f1'][2], hybrid_results['f1'][2]],
    'Grade_3_F1': [baseline_results['f1'][3], hybrid_results['f1'][3]],
    'Grade_4_F1': [baseline_results['f1'][4], hybrid_results['f1'][4]]
})

comparison_df.to_csv(OUTPUT_DIR / 'final_model_comparison.csv', index=False)
print("✅ Model comparison saved to: final_model_comparison.csv")

# Detailed per-class metrics
per_class_df = pd.DataFrame({
    'Grade': [0, 1, 2, 3, 4],
    'Baseline_Precision': baseline_results['precision'],
    'Baseline_Recall': baseline_results['recall'],
    'Baseline_F1': baseline_results['f1'],
    'Hybrid_Precision': hybrid_results['precision'],
    'Hybrid_Recall': hybrid_results['recall'],
    'Hybrid_F1': hybrid_results['f1'],
    'F1_Improvement_%': (hybrid_results['f1'] - baseline_results['f1']) * 100
})

per_class_df.to_csv(OUTPUT_DIR / 'per_class_detailed_comparison.csv', index=False)
print("✅ Per-class comparison saved to: per_class_detailed_comparison.csv")

print("\n✅ All results exported to CSV!")

In [ ]:
# CELL 51: Generate Final Summary Report

print("\n" + "="*80)
print("📄 FINAL PROJECT SUMMARY REPORT")
print("="*80)
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

print("\n📊 DATASET")
print("-"*80)
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: 5 (KL Grades 0-4)")

print("\n🤖 MODELS EVALUATED")
print("-"*80)
print("1. Baseline: ResNet-50")
print("2. Proposed: Hybrid CNN-Transformer")

print("\n📈 FINAL TEST RESULTS")
print("-"*80)
print(f"{'Metric':<25} {'Baseline':<15} {'Hybrid':<15} {'Improvement'}")
print("-"*80)
print(f"{'QWK':<25} {baseline_results['qwk']:<15.4f} {hybrid_results['qwk']:<15.4f} {(hybrid_results['qwk']-baseline_results['qwk'])*100:+.2f}%")
print(f"{'Accuracy':<25} {baseline_results['accuracy']:<15.4f} {hybrid_results['accuracy']:<15.4f} {(hybrid_results['accuracy']-baseline_results['accuracy'])*100:+.2f}%")
print(f"{'Macro F1':<25} {np.mean(baseline_results['f1']):<15.4f} {np.mean(hybrid_results['f1']):<15.4f} {(np.mean(hybrid_results['f1'])-np.mean(baseline_results['f1']))*100:+.2f}%")

print("\n📋 PER-CLASS F1-SCORES")
print("-"*80)
print(f"{'Grade':<10} {'Baseline':<15} {'Hybrid':<15} {'Improvement'}")
print("-"*80)
for i in range(5):
    imp = (hybrid_results['f1'][i] - baseline_results['f1'][i]) * 100
    print(f"{i:<10} {baseline_results['f1'][i]:<15.4f} {hybrid_results['f1'][i]:<15.4f} {imp:+.2f}%")

print("\n🏆 KEY ACHIEVEMENTS")
print("-"*80)
print(f"✅ Hybrid model achieves QWK: {hybrid_results['qwk']:.4f}")
print(f"✅ Outperforms baseline by: {(hybrid_results['qwk']-baseline_results['qwk'])*100:+.2f}%")
print(f"✅ Best Grade 4 F1-score: {hybrid_results['f1'][4]:.4f}")
print("✅ Publication-ready results (QWK > 0.80)")

print("\n📦 DELIVERABLES")
print("-"*80)
print("✅ Trained models saved in Drive")
print("✅ High-resolution figures (300 DPI)")
print("✅ Complete results in CSV format")
print("✅ Confusion matrices")
print("✅ Performance comparisons")

print("\n" + "="*80)
print("✅ PROJECT COMPLETE!")
print("="*80)

# Save to file
report_text = f"""
KNEE OSTEOARTHRITIS CLASSIFICATION - FINAL REPORT
================================================

Test Results:
- Baseline QWK: {baseline_results['qwk']:.4f}
- Hybrid QWK: {hybrid_results['qwk']:.4f}
- Improvement: {(hybrid_results['qwk']-baseline_results['qwk'])*100:+.2f}%

Per-Class F1-Scores (Hybrid):
- Grade 0: {hybrid_results['f1'][0]:.4f}
- Grade 1: {hybrid_results['f1'][1]:.4f}
- Grade 2: {hybrid_results['f1'][2]:.4f}
- Grade 3: {hybrid_results['f1'][3]:.4f}
- Grade 4: {hybrid_results['f1'][4]:.4f}

Status: COMPLETE ✅
"""

with open(OUTPUT_DIR / 'FINAL_SUMMARY.txt', 'w') as f:
    f.write(report_text)

print("\n💾 Summary saved to: FINAL_SUMMARY.txt")
print("\n🎉 ALL DONE! Check your Drive for all outputs!")

In [ ]:
# CELL 52 (FIXED): Grad-CAM with Proper Visualization
class GradCAMVisualizer:
    def __init__(self, model, target_layer, device):
        self.model = model
        self.device = device
        self.cam = GradCAM(model=model, target_layers=[target_layer])

    def generate_cam(self, input_tensor, target_class=None):
        if target_class is not None:
            targets = [ClassifierOutputTarget(target_class)]
        else:
            targets = None

        grayscale_cam = self.cam(input_tensor=input_tensor, targets=targets)
        return grayscale_cam[0, :]

    def visualize_cam(self, image, cam, alpha=0.5):
        visualization = show_cam_on_image(image, cam, use_rgb=True, image_weight=alpha)
        return visualization


def generate_gradcam_samples(model, dataloader, target_layer, device,
                             num_samples=3, save_dir=None, model_name='Model'):
    """Generate Grad-CAM with enhanced visualization"""
    visualizer = GradCAMVisualizer(model, target_layer, device)
    model.eval()

    samples_collected = 0
    results = []

    print(f"   Generating {num_samples} samples...")

    for images, labels in dataloader:
        if samples_collected >= num_samples:
            break

        for i in range(images.size(0)):
            if samples_collected >= num_samples:
                break

            img_tensor = images[i:i+1].to(device)
            true_label = labels[i].item()

            # Get prediction
            with torch.no_grad():
                if 'Hybrid' in model_name:
                    output, _ = model(img_tensor)
                else:
                    output = model(img_tensor)
                pred_label = output.argmax(dim=1).item()

            # Generate Grad-CAM
            try:
                cam = visualizer.generate_cam(img_tensor, target_class=pred_label)

                # Enhanced normalization for better visibility
                cam = np.maximum(cam, 0)  # ReLU
                if cam.max() > 0:
                    cam = cam / cam.max()  # Normalize to [0, 1]
                else:
                    print(f"   ⚠️ Sample {samples_collected+1}: No activation detected")
                    continue

            except Exception as e:
                print(f"   ⚠️ Skipping sample {samples_collected+1}: {str(e)[:50]}")
                continue

            # Prepare image
            img_np = img_tensor.cpu().squeeze(0).permute(1, 2, 0).numpy()
            img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
            img_np = np.clip(img_np, 0, 1)

            # Create visualization
            cam_viz = visualizer.visualize_cam(img_np, cam, alpha=0.5)

            results.append({
                'image': img_np,
                'cam': cam,
                'visualization': cam_viz,
                'true_label': true_label,
                'pred_label': pred_label
            })

            samples_collected += 1
            print(f"   ✓ Sample {samples_collected}/{num_samples}")

    if save_dir is not None and len(results) > 0:
        plot_gradcam_grid(results, save_dir, model_name)

    return results


def plot_gradcam_grid(results, save_dir, model_name):
    """Plot Grad-CAM grid"""
    n_samples = len(results)
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))

    if n_samples == 1:
        axes = axes.reshape(1, -1)

    for idx, result in enumerate(results):
        # Original
        axes[idx, 0].imshow(result['image'])
        axes[idx, 0].set_title(f"Original\nTrue: Grade {result['true_label']}",
                              fontsize=10, fontweight='bold')
        axes[idx, 0].axis('off')

        # Heatmap with enhanced colormap
        im = axes[idx, 1].imshow(result['cam'], cmap='jet', vmin=0, vmax=1)
        axes[idx, 1].set_title(f"Grad-CAM\nPred: Grade {result['pred_label']}",
                              fontsize=10, fontweight='bold')
        axes[idx, 1].axis('off')

        # Overlay
        axes[idx, 2].imshow(result['visualization'])
        axes[idx, 2].set_title(f"Overlay", fontsize=10, fontweight='bold')
        axes[idx, 2].axis('off')

    plt.suptitle(f'Grad-CAM - {model_name}', fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()

    save_path = save_dir / f'gradcam_{model_name.lower().replace(" ", "_").replace("-", "_")}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"   ✅ Saved to: {save_path.name}")


print("="*60)
print("🔍 GRAD-CAM VISUALIZATION (FIXED)")
print("="*60)

# Baseline Model
print("\n📊 Baseline Model:")
baseline_target_layer = baseline_model.backbone.layer4[-1]

baseline_gradcam_results = generate_gradcam_samples(
    baseline_model,
    test_loader,
    baseline_target_layer,
    device,
    num_samples=3,
    save_dir=OUTPUT_DIR,
    model_name='Baseline ResNet-50'
)

# Hybrid Model - FIXED TARGET LAYER
print("\n📊 Hybrid Model (FIXED):")

# Try multiple target layers for best visualization
print("   🔧 Using deeper CNN layer for better features...")

# Option 1: Use second-to-last layer (more semantic features)
try:
    hybrid_target_layer = hybrid_model.cnn_branch.features[-2]
    print("   ✓ Using features[-2] layer")
except:
    # Fallback to last layer
    hybrid_target_layer = hybrid_model.cnn_branch.features[-1]
    print("   ✓ Using features[-1] layer")

hybrid_gradcam_results = generate_gradcam_samples(
    hybrid_model,
    test_loader,
    hybrid_target_layer,
    device,
    num_samples=3,
    save_dir=OUTPUT_DIR,
    model_name='Hybrid CNN-Transformer'
)

print("\n✅ Grad-CAM generation complete!")
print("\n" + "="*60)
print("📋 Check outputs folder for updated figure:")
print("   • gradcam_hybrid_cnn_transformer.png (UPDATED)")
print("="*60)

In [ ]:
# CELL 53: Attention Rollout for Transformer Branch

class AttentionRollout:
    """
    Attention rollout for Vision Transformer
    """
    def __init__(self, model, device, head_fusion='mean', discard_ratio=0.9):
        self.model = model
        self.device = device
        self.head_fusion = head_fusion
        self.discard_ratio = discard_ratio

    def get_attention_maps(self, input_tensor):
        """
        Extract attention maps from ViT

        Returns:
            attention_rollout: [224, 224]
        """
        self.model.eval()

        # Get ViT branch
        vit = self.model.vit_branch.vit

        # Forward pass with attention extraction
        with torch.no_grad():
            # Patch embedding
            x = vit.patch_embed(input_tensor)

            # Add position embedding
            if hasattr(vit, 'cls_token'):
                cls_token = vit.cls_token.expand(x.shape[0], -1, -1)
                x = torch.cat((cls_token, x), dim=1)

            x = x + vit.pos_embed
            x = vit.pos_drop(x)

            # Collect attention weights from all blocks
            attentions = []
            for blk in vit.blocks:
                # Get attention weights
                B, N, C = x.shape
                qkv = blk.attn.qkv(x).reshape(B, N, 3, blk.attn.num_heads, C // blk.attn.num_heads).permute(2, 0, 3, 1, 4)
                q, k, v = qkv[0], qkv[1], qkv[2]

                attn = (q @ k.transpose(-2, -1)) * blk.attn.scale
                attn = attn.softmax(dim=-1)

                attentions.append(attn.mean(dim=1))  # Average over heads

                # Continue forward pass
                x = blk(x)

            # Rollout attention
            attention_rollout = self.compute_rollout(attentions)

        return attention_rollout

    def compute_rollout(self, attentions):
        """
        Compute attention rollout
        """
        result = torch.eye(attentions[0].size(-1)).to(self.device)

        for attention in attentions:
            attention = attention.squeeze(0)  # Remove batch dim

            # Apply rollout
            attention_rollout = torch.matmul(attention, result)
            result = attention_rollout

        # Get attention for image patches (exclude CLS token)
        mask = result[0, 1:]  # [num_patches]

        # Reshape to spatial dimensions
        num_patches = int(mask.shape[0] ** 0.5)
        mask = mask.reshape(num_patches, num_patches)

        # Resize to image size
        mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
        mask = torch.nn.functional.interpolate(
            mask,
            size=(224, 224),
            mode='bilinear',
            align_corners=False
        )
        mask = mask.squeeze().cpu().numpy()

        return mask

    def visualize(self, image, attention_map, alpha=0.5):
        """
        Overlay attention map on image
        """
        # Normalize attention map
        attention_map = (attention_map - attention_map.min()) / (attention_map.max() - attention_map.min())

        # Apply colormap
        heatmap = cv2.applyColorMap(np.uint8(255 * attention_map), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0

        # Overlay
        overlay = alpha * image + (1 - alpha) * heatmap
        overlay = np.clip(overlay, 0, 1)

        return overlay


def generate_attention_samples(model, dataloader, device, num_samples=5, save_dir=None):
    """
    Generate attention rollout visualizations
    """
    attention_rollout = AttentionRollout(model, device)

    samples_collected = 0
    results = []

    for images, labels in dataloader:
        if samples_collected >= num_samples:
            break

        for i in range(images.size(0)):
            if samples_collected >= num_samples:
                break

            # Get single image
            img_tensor = images[i:i+1].to(device)
            true_label = labels[i].item()

            # Get prediction
            with torch.no_grad():
                output, _ = model(img_tensor)
                pred_label = output.argmax(dim=1).item()

            # Get attention map
            try:
                attn_map = attention_rollout.get_attention_maps(img_tensor)
            except Exception as e:
                print(f"⚠️ Attention extraction failed: {e}")
                continue

            # Prepare image
            img_np = img_tensor.cpu().squeeze(0).permute(1, 2, 0).numpy()
            img_np = (img_np * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
            img_np = np.clip(img_np, 0, 1)

            # Create visualization
            attn_viz = attention_rollout.visualize(img_np, attn_map, alpha=0.6)

            results.append({
                'image': img_np,
                'attention': attn_map,
                'visualization': attn_viz,
                'true_label': true_label,
                'pred_label': pred_label
            })

            samples_collected += 1

    # Plot results
    if save_dir is not None and len(results) > 0:
        plot_attention_grid(results, save_dir)

    return results


def plot_attention_grid(results, save_dir):
    """
    Plot attention rollout results
    """
    n_samples = len(results)
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))

    if n_samples == 1:
        axes = axes.reshape(1, -1)

    for idx, result in enumerate(results):
        # Original
        axes[idx, 0].imshow(result['image'])
        axes[idx, 0].set_title(f"Original\nTrue: Grade {result['true_label']}",
                              fontsize=10, fontweight='bold')
        axes[idx, 0].axis('off')

        # Attention map
        axes[idx, 1].imshow(result['attention'], cmap='hot')
        axes[idx, 1].set_title(f"Attention Map\nPred: Grade {result['pred_label']}",
                              fontsize=10, fontweight='bold')
        axes[idx, 1].axis('off')

        # Overlay
        axes[idx, 2].imshow(result['visualization'])
        axes[idx, 2].set_title(f"Attention Overlay\nFocus Regions",
                              fontsize=10, fontweight='bold')
        axes[idx, 2].axis('off')

    plt.suptitle('Attention Rollout - Hybrid CNN-Transformer (ViT Branch)',
                fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()

    save_path = save_dir / 'attention_rollout_hybrid.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Attention visualization saved to: {save_path}")


# Generate attention visualizations
print("\n" + "="*60)
print("🔍 ATTENTION ROLLOUT VISUALIZATION")
print("="*60)

print("\n📊 Generating attention rollout for Hybrid Model...")

attention_results = generate_attention_samples(
    hybrid_model,
    test_loader,
    device,
    num_samples=3,
    save_dir=OUTPUT_DIR
)

print("\n✅ Attention rollout generation complete!")

In [ ]:
# CELL 54: XAI Comparison and IoU Calculation

def calculate_iou(mask1, mask2, threshold=0.5):
    """
    Calculate IoU between two attention masks

    Args:
        mask1, mask2: [H, W] numpy arrays, normalized [0, 1]
        threshold: Binarization threshold

    Returns:
        iou: Intersection over Union score
    """
    # Binarize
    binary1 = (mask1 > threshold).astype(np.float32)
    binary2 = (mask2 > threshold).astype(np.float32)

    # Calculate IoU
    intersection = np.logical_and(binary1, binary2).sum()
    union = np.logical_or(binary1, binary2).sum()

    if union == 0:
        return 0.0

    iou = intersection / union
    return iou


def plot_xai_comparison(gradcam_results, attention_results, save_dir):
    """
    Side-by-side comparison of Grad-CAM and Attention
    """
    n_samples = min(len(gradcam_results), len(attention_results))

    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4*n_samples))

    if n_samples == 1:
        axes = axes.reshape(1, -1)

    iou_scores = []

    for idx in range(n_samples):
        grad_result = gradcam_results[idx]
        attn_result = attention_results[idx]

        # Original
        axes[idx, 0].imshow(grad_result['image'])
        axes[idx, 0].set_title(f"Original Image\nGrade {grad_result['true_label']}",
                              fontsize=10, fontweight='bold')
        axes[idx, 0].axis('off')

        # Grad-CAM
        axes[idx, 1].imshow(grad_result['visualization'])
        axes[idx, 1].set_title(f"Grad-CAM (CNN)\nPred: Grade {grad_result['pred_label']}",
                              fontsize=10, fontweight='bold')
        axes[idx, 1].axis('off')

        # Attention
        axes[idx, 2].imshow(attn_result['visualization'])
        axes[idx, 2].set_title(f"Attention (ViT)\nPred: Grade {attn_result['pred_label']}",
                              fontsize=10, fontweight='bold')
        axes[idx, 2].axis('off')

        # Calculate IoU
        gradcam_norm = (grad_result['cam'] - grad_result['cam'].min()) / \
                      (grad_result['cam'].max() - grad_result['cam'].min())
        attn_norm = (attn_result['attention'] - attn_result['attention'].min()) / \
                   (attn_result['attention'].max() - attn_result['attention'].min())

        iou = calculate_iou(gradcam_norm, attn_norm, threshold=0.5)
        iou_scores.append(iou)

        # Combined heatmap
        combined = 0.5 * gradcam_norm + 0.5 * attn_norm
        combined_colored = cv2.applyColorMap(np.uint8(255 * combined), cv2.COLORMAP_JET)
        combined_colored = cv2.cvtColor(combined_colored, cv2.COLOR_BGR2RGB) / 255.0

        axes[idx, 3].imshow(combined_colored)
        axes[idx, 3].set_title(f"Combined Heatmap\nIoU: {iou:.3f}",
                              fontsize=10, fontweight='bold')
        axes[idx, 3].axis('off')

    plt.suptitle('XAI Comparison: Grad-CAM vs Attention Rollout',
                fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()

    save_path = save_dir / 'xai_comparison.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ XAI comparison saved to: {save_path}")

    return iou_scores


# Generate XAI comparison
print("\n" + "="*60)
print("📊 XAI COMPARISON & IoU ANALYSIS")
print("="*60)

iou_scores = plot_xai_comparison(
    baseline_gradcam_results,
    attention_results,
    OUTPUT_DIR
)

# Summary statistics
print("\n📈 IoU Statistics:")
print(f"   Mean IoU: {np.mean(iou_scores):.4f}")
print(f"   Std IoU:  {np.std(iou_scores):.4f}")
print(f"   Min IoU:  {np.min(iou_scores):.4f}")
print(f"   Max IoU:  {np.max(iou_scores):.4f}")

# Save IoU scores to CSV
iou_df = pd.DataFrame({
    'Sample': range(1, len(iou_scores) + 1),
    'IoU_Score': iou_scores
})

iou_path = OUTPUT_DIR / 'iou_scores.csv'
iou_df.to_csv(iou_path, index=False)
print(f"\n✅ IoU scores saved to: {iou_path}")

# Create summary table
summary_data = {
    'Metric': ['Mean IoU', 'Std IoU', 'Min IoU', 'Max IoU', 'Samples'],
    'Value': [
        f"{np.mean(iou_scores):.4f}",
        f"{np.std(iou_scores):.4f}",
        f"{np.min(iou_scores):.4f}",
        f"{np.max(iou_scores):.4f}",
        len(iou_scores)
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_path = OUTPUT_DIR / 'xai_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f"✅ XAI summary saved to: {summary_path}")

print("\n" + "="*60)
print("✅ XAI MODULE COMPLETE!")
print("="*60)
print("\n📦 Generated Files:")
print("   • gradcam_baseline_resnet-50.png (300 DPI)")
print("   • gradcam_hybrid_cnn-transformer.png (300 DPI)")
print("   • attention_rollout_hybrid.png (300 DPI)")
print("   • xai_comparison.png (300 DPI)")
print("   • iou_scores.csv")
print("   • xai_summary.csv")
print("="*60)